In [1]:
# LABELING 01 — ENVIRONMENT & SOURCE VALIDATION
# OFFLINE ONLY | NO AUTO LABELING
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

LABELING_DIR = ROOT / "03_Labeling"
INPUT_DIR = LABELING_DIR / "inputs"
CODEBOOK_DIR = LABELING_DIR / "codebook"
OUTPUT_DIR = LABELING_DIR / "outputs"

CODEBOOK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_PATH = INPUT_DIR / "labeling_all_381.csv"
MODEL_PATH = INPUT_DIR / "labeling_candidates_348.csv"
MANIFEST_PATH = INPUT_DIR / "cleaning_handoff_manifest.json"

for path in [ALL_PATH, MODEL_PATH, MANIFEST_PATH]:
    assert path.is_file(), f"Missing input: {path}"

df_all = pd.read_csv(
    ALL_PATH,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

df_model = pd.read_csv(
    MODEL_PATH,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

manifest = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)

assert len(df_all) == 381
assert len(df_model) == 348

assert df_all["id"].is_unique
assert df_model["id"].is_unique

assert set(df_model["id"]).issubset(set(df_all["id"]))

assert df_all["sentiment_label"].eq("").all()
assert df_model["sentiment_label"].eq("").all()

assert df_all["text_original"].str.strip().ne("").all()
assert df_all["text_semantic_verified"].str.strip().ne("").all()

assert manifest["status"] == "READY_FOR_LABELING"
assert manifest["total_eligible_tweets"] == 381
assert manifest["model_candidates"] == 348

EXPECTED_CASES = {
    "lombok_2018": 114,
    "palu_2018": 118,
    "sumatra_2025": 149
}

actual_cases = df_all["case"].value_counts().to_dict()
assert actual_cases == EXPECTED_CASES, actual_cases

# Check semua teks kandidat masih identik dengan
# sumber ALL_381 berdasarkan ID.
source_text = df_all.set_index("id")["text_linguistic"]
candidate_text = df_model.set_index("id")["text_linguistic"]

assert candidate_text.equals(
    source_text.loc[candidate_text.index]
)

print("=" * 60)
print("LABELING 01 — SOURCE VALIDATION")
print("=" * 60)
print("Total eligible tweets :", len(df_all))
print("Model candidates      :", len(df_model))
print("Unassigned labels     :", df_all["sentiment_label"].eq("").sum())
print("Cases                 :", actual_cases)
print("Source integrity      : PASS")
print("No data modification.")


LABELING 01 — SOURCE VALIDATION
Total eligible tweets : 381
Model candidates      : 348
Unassigned labels     : 381
Cases                 : {'sumatra_2025': 149, 'palu_2018': 118, 'lombok_2018': 114}
Source integrity      : PASS
No data modification.


In [2]:
# LABELING 02 — SENTIMENT CODEBOOK v1.0
# CREATE DOCUMENTATION | NO SENTIMENT INFERENCE
# ============================================================

CODEBOOK_PATH = (
    CODEBOOK_DIR / "sentiment_codebook_v1.md"
)

assert not CODEBOOK_PATH.exists(), (
    f"Codebook sudah ada: {CODEBOOK_PATH}"
)

codebook = """# Sentiment Labeling Codebook v1.0

## 1. Tujuan
Mengidentifikasi sikap penulis terhadap penetapan status
bencana nasional pada Lombok 2018, Palu 2018,
dan Sumatera 2025.

## 2. Unit Analisis
Satu postingan/tweet dengan tweet ID unik.

Setiap tweet diberi tepat satu label akhir.

## 3. Objek Sentimen
Kebijakan atau keputusan penetapan status bencana nasional.

Penilaian bukan didasarkan pada sentimen umum terhadap
bencana, korban, maupun pemerintah.

## 4. Kategori

### POSITIF
Dukungan, persetujuan, desakan, tuntutan, atau kritik
yang secara jelas mendukung penetapan status
bencana nasional.

### NEGATIF
Penolakan, ketidaksetujuan, atau pernyataan bahwa
penetapan status bencana nasional tidak diperlukan.

### NETRAL
Berita, laporan, pertanyaan informatif, kutipan tanpa
persetujuan penulis, atau unggahan tanpa sikap jelas
terhadap penetapan status bencana nasional.

## 5. Prinsip Anotasi
1. Utamakan maksud penulis, bukan hanya keyword.
2. Jangan samakan kesedihan akibat bencana dengan POSITIF.
3. Jangan samakan kritik kepada pemerintah dengan POSITIF
   kecuali terkait dukungan terhadap penetapan status.
4. Bedakan opini penulis dari kutipan atau laporan berita.
5. Pertanyaan retoris dinilai dari konteks.
6. Jika penulis mendukung sekaligus menolak dan sikap
   dominan tidak jelas, tandai untuk review.
7. Gunakan text_original dan text_semantic_verified
   untuk memahami konteks.
8. Jangan menggunakan label hasil prediksi model
   sebagai pengganti keputusan anotator.
9. Jangan mengubah teks sumber ketika memberikan label.
10. Label akhir hanya POSITIF, NEGATIF, atau NETRAL.

## 6. Status Review
PENDING: belum dianotasi.
LABELED: label telah diberikan.
AMBIGUOUS: membutuhkan adjudikasi.
ADJUDICATED: kasus telah diputuskan melalui review.

AMBIGUOUS bukan kelas sentimen keempat.

## 7. Contoh Ilustratif
- "Segera tetapkan Palu sebagai bencana nasional."
  Label: POSITIF

- "Tidak perlu menetapkan status bencana nasional."
  Label: NEGATIF

- "Pemerintah menjelaskan alasan status belum ditetapkan."
  Label: NETRAL

- "Mengapa Lombok belum menjadi bencana nasional?"
  Label: NETRAL jika pertanyaan informatif.
  Review apabila konteks mengisyaratkan desakan.

## 8. Konsistensi
- Satu ID harus memiliki satu label final.
- Konten identik perlu diperiksa konsistensinya.
- Label untuk kandidat modeling diturunkan dari ALL_381.
- Semua keputusan ambigu harus diselesaikan sebelum
  ekspor dataset final.

## 9. Versi
Version: 1.0
Status: DRAFT_FOR_PILOT
"""

CODEBOOK_PATH.write_text(
    codebook,
    encoding="utf-8"
)

required_sections = [
    "POSITIF", "NEGATIF", "NETRAL",
    "AMBIGUOUS", "text_original",
    "text_semantic_verified",
    "DRAFT_FOR_PILOT"
]

saved = CODEBOOK_PATH.read_text(encoding="utf-8")

assert all(
    section in saved for section in required_sections
)

print("=" * 60)
print("LABELING 02 — CODEBOOK CREATION")
print("=" * 60)
print("File       :", CODEBOOK_PATH)
print("Version    : 1.0")
print("Sentiments : POSITIF, NEGATIF, NETRAL")
print("Status     : DRAFT_FOR_PILOT")
print("CODEBOOK VALIDATION: PASS")


LABELING 02 — CODEBOOK CREATION
File       : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\codebook\sentiment_codebook_v1.md
Version    : 1.0
Sentiments : POSITIF, NEGATIF, NETRAL
Status     : DRAFT_FOR_PILOT
CODEBOOK VALIDATION: PASS


In [3]:
# LABELING 03 — CODEBOOK CONSISTENCY TEST
# ILLUSTRATIVE CASES ONLY | NO MODEL PREDICTION
# ============================================================

examples = [
    {
        "text": "Tetapkan banjir Sumatera sebagai bencana nasional.",
        "expected_label": "POSITIF",
        "reason": "Desakan langsung"
    },
    {
        "text": "Palu seharusnya segera berstatus bencana nasional.",
        "expected_label": "POSITIF",
        "reason": "Dukungan jelas"
    },
    {
        "text": "Tidak perlu menaikkan status menjadi bencana nasional.",
        "expected_label": "NEGATIF",
        "reason": "Penolakan eksplisit"
    },
    {
        "text": "Status nasional bukan langkah yang diperlukan.",
        "expected_label": "NEGATIF",
        "reason": "Menilai penetapan tidak perlu"
    },
    {
        "text": "BNPB membahas status bencana nasional Lombok.",
        "expected_label": "NETRAL",
        "reason": "Informasi tanpa sikap penulis"
    },
    {
        "text": "Presiden menjelaskan alasan status belum ditetapkan.",
        "expected_label": "NETRAL",
        "reason": "Laporan pernyataan"
    },
    {
        "text": "Mengapa status bencana nasional belum ditetapkan?",
        "expected_label": "REVIEW",
        "reason": "Pertanyaan berpotensi ambigu"
    },
]

example_df = pd.DataFrame(examples)

VALID_LABELS = {"POSITIF", "NEGATIF", "NETRAL"}
VALID_TEST_DECISIONS = VALID_LABELS | {"REVIEW"}

assert example_df["expected_label"].isin(
    VALID_TEST_DECISIONS
).all()

assert not example_df["text"].duplicated().any()

print("CODEBOOK EXAMPLE CHECK: PASS")
print("Example count:", len(example_df))
print("Final classes:", sorted(VALID_LABELS))
print("Review is a workflow status, not a fourth class.")

display(example_df)


CODEBOOK EXAMPLE CHECK: PASS
Example count: 7
Final classes: ['NEGATIF', 'NETRAL', 'POSITIF']
Review is a workflow status, not a fourth class.


,text,expected_label,reason
0,Tetapkan banjir Sumatera sebagai bencana nasio...,POSITIF,Desakan langsung
1,Palu seharusnya segera berstatus bencana nasio...,POSITIF,Dukungan jelas
2,Tidak perlu menaikkan status menjadi bencana n...,NEGATIF,Penolakan eksplisit
3,Status nasional bukan langkah yang diperlukan.,NEGATIF,Menilai penetapan tidak perlu
4,BNPB membahas status bencana nasional Lombok.,NETRAL,Informasi tanpa sikap penulis
5,Presiden menjelaskan alasan status belum ditet...,NETRAL,Laporan pernyataan
6,Mengapa status bencana nasional belum ditetapkan?,REVIEW,Pertanyaan berpotensi ambigu


In [4]:
# LABELING 04 — STRATIFIED PILOT ANNOTATION PREPARATION
# 30 TWEETS | 10 PER CASE | OFFLINE ONLY
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

assert len(df_all) == 381
assert df_all["id"].is_unique

PILOT_SEED = 42
PILOT_PER_CASE = 10

PILOT_DIR = OUTPUT_DIR / "pilot"
PILOT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_CSV = PILOT_DIR / "pilot_annotation_30.csv"
PILOT_XLSX = PILOT_DIR / "pilot_annotation_30.xlsx"
PILOT_MANIFEST = PILOT_DIR / "pilot_manifest.json"

for path in [PILOT_CSV, PILOT_XLSX, PILOT_MANIFEST]:
    if path.exists():
        raise FileExistsError(
            f"Output sudah tersedia: {path}"
        )

# ------------------------------------------------------------
# A. Pilih satu teks per grup terlebih dahulu
# agar pilot berfokus pada variasi konten.
# ------------------------------------------------------------

pool = df_all.loc[
    df_all["eligible_for_model_candidate"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
].copy()

assert len(pool) == 348

# Untuk memastikan ada variasi konten dan
# sebagian tweet sulit juga terwakili,
# pilih sampel secara deterministik.
pilot_parts = []

for case in sorted(pool["case"].unique()):
    subset = pool.loc[pool["case"] == case]

    assert len(subset) >= PILOT_PER_CASE

    selected = subset.sample(
        n=PILOT_PER_CASE,
        random_state=PILOT_SEED
    )

    pilot_parts.append(selected)

pilot = pd.concat(
    pilot_parts,
    ignore_index=True
)

assert len(pilot) == 30
assert pilot["id"].is_unique

assert pilot["case"].value_counts().to_dict() == {
    "lombok_2018": 10,
    "palu_2018": 10,
    "sumatra_2025": 10
}

# ------------------------------------------------------------
# B. Buat kolom anotasi manual
# ------------------------------------------------------------

pilot = pilot[
    [
        "case",
        "id",
        "text_original",
        "text_semantic_verified",
        "text_linguistic",
        "model_selection_status",
        "final_text_hash"
    ]
].copy()

pilot["sentiment_label"] = ""
pilot["annotation_status"] = "PENDING"
pilot["label_notes"] = ""
pilot["annotator"] = ""
pilot["labeled_at"] = ""

# ------------------------------------------------------------
# C. Simpan CSV
# ------------------------------------------------------------

pilot.to_csv(
    PILOT_CSV,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# D. Simpan workbook Excel
# ------------------------------------------------------------

from openpyxl.styles import Font, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

with pd.ExcelWriter(
    PILOT_XLSX,
    engine="openpyxl"
) as writer:

    pilot.to_excel(
        writer,
        sheet_name="PILOT_30",
        index=False
    )

    ws = writer.sheets["PILOT_30"]
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = ws.dimensions

    headers = {
        cell.value: cell.column_letter
        for cell in ws[1]
    }

    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(wrap_text=True)

    for column in ws.columns:
        name = str(column[0].value)
        letter = column[0].column_letter

        ws.column_dimensions[letter].width = (
            75 if name.startswith("text_") else 25
        )

        for cell in column[1:]:
            if isinstance(cell.value, str):
                cell.data_type = "s"

            if name == "id":
                cell.number_format = "@"

    # Dropdown sentimen:
    sentiment_col = headers["sentiment_label"]
    sentiment_validation = DataValidation(
        type="list",
        formula1='"POSITIF,NEGATIF,NETRAL"',
        allow_blank=True
    )
    ws.add_data_validation(sentiment_validation)
    sentiment_validation.add(
        f"{sentiment_col}2:{sentiment_col}31"
    )

    # Dropdown status:
    status_col = headers["annotation_status"]
    status_validation = DataValidation(
        type="list",
        formula1='"PENDING,LABELED,AMBIGUOUS,ADJUDICATED"',
        allow_blank=False
    )
    ws.add_data_validation(status_validation)
    status_validation.add(
        f"{status_col}2:{status_col}31"
    )

# ------------------------------------------------------------
# E. Manifest untuk reproduktibilitas
# ------------------------------------------------------------

pilot_manifest = {
    "stage": "03_Labeling",
    "part": "Pilot Annotation Preparation",
    "codebook_version": "1.0",
    "codebook_status": "DRAFT_FOR_PILOT",
    "seed": PILOT_SEED,
    "sampling_method": "stratified_random_by_case",
    "sample_size": len(pilot),
    "sample_per_case": PILOT_PER_CASE,
    "labels_assigned": 0,
    "status": "READY_FOR_PILOT_ANNOTATION"
}

PILOT_MANIFEST.write_text(
    json.dumps(
        pilot_manifest,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# F. Audit
# ------------------------------------------------------------

print("=" * 64)
print("LABELING 04 — PILOT ANNOTATION PREPARATION")
print("=" * 64)

print("Source tweets       :", len(df_all))
print("Model candidate pool:", len(pool))
print("Pilot sample        :", len(pilot))
print("Pilot labels filled :", 0)

print("\nDistribution:")
display(pilot["case"].value_counts())

print("\nOutput files:")
print(PILOT_CSV)
print(PILOT_XLSX)
print(PILOT_MANIFEST)

print("\nLABELING 04 PILOT PREPARATION: PASS")


LABELING 04 — PILOT ANNOTATION PREPARATION
Source tweets       : 381
Model candidate pool: 348
Pilot sample        : 30
Pilot labels filled : 0

Distribution:


case
lombok_2018     10
palu_2018       10
sumatra_2025    10
Name: count, dtype: int64


Output files:
C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\pilot\pilot_annotation_30.csv
C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\pilot\pilot_annotation_30.xlsx
C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\pilot\pilot_manifest.json

LABELING 04 PILOT PREPARATION: PASS


In [5]:
# LABELING 05 — PILOT ANNOTATION VALIDATION
# 30 TWEETS | SOURCE INTEGRITY | ADJUDICATION FLAGS
# OFFLINE ONLY | NO AUTOMATIC RELABELING
# ============================================================

import pandas as pd
from pathlib import Path
import json

PILOT_DIR = OUTPUT_DIR / "pilot"
PILOT_XLSX = PILOT_DIR / "pilot_annotation_30.xlsx"
PILOT_CSV = PILOT_DIR / "pilot_annotation_30.csv"

AUDIT_CSV = PILOT_DIR / "pilot_validation_audit.csv"
AUDIT_JSON = PILOT_DIR / "pilot_validation_summary.json"

for p in [AUDIT_CSV, AUDIT_JSON]:
    if p.exists():
        raise FileExistsError(f"Output already exists: {p}")

# A. Read source snapshot and manually edited workbook
source = pd.read_csv(
    PILOT_CSV,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

reviewed = pd.read_excel(
    PILOT_XLSX,
    sheet_name="PILOT_30",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

assert len(source) == len(reviewed) == 30
assert source["id"].is_unique
assert reviewed["id"].is_unique

# B. Verify identities and immutable columns
assert set(source["id"]) == set(reviewed["id"])

immutable_cols = [
    "case",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "model_selection_status",
    "final_text_hash"
]

source_indexed = source.set_index("id")
reviewed_indexed = reviewed.set_index("id")

for col in immutable_cols:
    expected = source_indexed[col].astype("string")
    actual = reviewed_indexed[col].astype("string")

    assert actual.sort_index().equals(
        expected.sort_index()
    ), f"Source integrity mismatch: {col}"

# C. Normalize annotation fields for validation
reviewed["sentiment_label"] = (
    reviewed["sentiment_label"]
    .astype("string").str.strip().str.upper()
)

reviewed["annotation_status"] = (
    reviewed["annotation_status"]
    .astype("string").str.strip().str.upper()
)

allowed_labels = {"POSITIF", "NEGATIF", "NETRAL"}

assert reviewed["sentiment_label"].isin(
    allowed_labels
).all(), "Invalid or missing sentiment label."

assert reviewed["annotation_status"].eq(
    "LABELED"
).all(), "Some rows are not LABELED."

for col in ["label_notes", "annotator", "labeled_at"]:
    assert reviewed[col].astype("string").str.strip().ne(
        ""
    ).all(), f"Missing values in: {col}"

dates = pd.to_datetime(
    reviewed["labeled_at"],
    errors="coerce"
)

assert dates.notna().all(), "Invalid annotation dates."

# D. Candidate semantic review:
# Manual audit targets, NOT automatic corrections.
REVIEW_IDS = {
    "1028865436794810368": (
        "Penolakan atau kritik terhadap pemerintah?"
    ),
    "1046304416356130816": (
        "Pertanyaan atau dukungan eksplisit?"
    ),
    "1047236117546774528": (
        "Judul berita atau dukungan penulis?"
    ),
    "1996474933816529181": (
        "Laporan desakan pihak lain atau sikap penulis?"
    )
}

reviewed["needs_adjudication"] = (
    reviewed["id"].isin(REVIEW_IDS)
)

reviewed["adjudication_reason"] = (
    reviewed["id"].map(REVIEW_IDS).fillna("")
)

assert reviewed["needs_adjudication"].sum() == 4

# E. Distribution
distribution = (
    reviewed.groupby(
        ["case", "sentiment_label"]
    )
    .size()
    .unstack(fill_value=0)
)

print("LABEL DISTRIBUTION BY CASE")
display(distribution)

print("\nSEMANTIC ADJUDICATION CANDIDATES")
display(
    reviewed.loc[
        reviewed["needs_adjudication"],
        [
            "case",
            "id",
            "text_original",
            "text_semantic_verified",
            "sentiment_label",
            "label_notes",
            "adjudication_reason"
        ]
    ]
)

# F. Save read-only derived audit report
reviewed.to_csv(
    AUDIT_CSV,
    index=False,
    encoding="utf-8-sig"
)

summary = {
    "total_pilot_tweets": len(reviewed),
    "labeled": int(
        reviewed["annotation_status"].eq("LABELED").sum()
    ),
    "positive": int(
        reviewed["sentiment_label"].eq("POSITIF").sum()
    ),
    "negative": int(
        reviewed["sentiment_label"].eq("NEGATIF").sum()
    ),
    "neutral": int(
        reviewed["sentiment_label"].eq("NETRAL").sum()
    ),
    "source_integrity": "PASS",
    "manual_adjudication_candidates": int(
        reviewed["needs_adjudication"].sum()
    ),
    "status": "ADJUDICATION_PENDING"
}

AUDIT_JSON.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n" + "=" * 62)
print("LABELING 05 — PILOT VALIDATION")
print("=" * 62)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nValidation completed.")
print("Original files unchanged.")
print("No automatic label modification.")


LABEL DISTRIBUTION BY CASE


sentiment_label,NEGATIF,NETRAL,POSITIF
case,,,
lombok_2018,2,5,3
palu_2018,1,7,2
sumatra_2025,0,4,6



SEMANTIC ADJUDICATION CANDIDATES


,case,id,text_original,text_semantic_verified,sentiment_label,label_notes,adjudication_reason
6,lombok_2018,1028865436794810368,"Sampai saat ini, bantuan gempa Lombok, masih t...","Sampai saat ini, bantuan gempa Lombok, masih t...",NEGATIF,menyatakan bahwa pemerintah masih belum lumpuh...,Penolakan atau kritik terhadap pemerintah?
15,palu_2018,1046304416356130816,Apakah bapak Presiden juga\nsiap membantu kami...,Apakah bapak Presiden juga siap membantu kami ...,POSITIF,menunjukkan pertanyaan terhadap kesiapan pemer...,Pertanyaan atau dukungan eksplisit?
17,palu_2018,1047236117546774528,PKS Desak Pemerintah Tetapkan Gempa Palu Benca...,PKS Desak Pemerintah Tetapkan Gempa Palu Benca...,POSITIF,menyatakan desakan kepada pemerintah,Judul berita atau dukungan penulis?
24,sumatra_2025,1996474933816529181,Melihat banjir bandang yang memporak-porandaka...,Melihat banjir bandang yang memporak-porandaka...,POSITIF,berisi informasi yang menyatakan desakan untuk...,Laporan desakan pihak lain atau sikap penulis?



LABELING 05 — PILOT VALIDATION
total_pilot_tweets: 30
labeled: 30
positive: 11
negative: 3
neutral: 16
source_integrity: PASS
manual_adjudication_candidates: 4
status: ADJUDICATION_PENDING

Validation completed.
Original files unchanged.
No automatic label modification.


In [6]:
# LABELING 06 — PILOT ADJUDICATION PREPARATION
# 4 FLAGGED TWEETS | NO AUTOMATIC RELABELING
# ============================================================

from pathlib import Path
import pandas as pd
from openpyxl.styles import Font, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

PILOT_DIR = OUTPUT_DIR / "pilot"
AUDIT_PATH = PILOT_DIR / "pilot_validation_audit.csv"
ADJUDICATION_PATH = (
    PILOT_DIR / "pilot_adjudication_4.xlsx"
)

assert AUDIT_PATH.exists(), "Audit Part4 tidak ditemukan."
assert not ADJUDICATION_PATH.exists(), (
    "Workbook adjudication sudah ada. Jangan overwrite."
)

audit = pd.read_csv(
    AUDIT_PATH,
    dtype={"id": "string"},
    encoding="utf-8-sig",
    keep_default_na=False
)

flagged = audit.loc[
    audit["needs_adjudication"]
    .astype(str).str.lower().eq("true")
].copy()

assert len(flagged) == 4
assert flagged["id"].is_unique

columns = [
    "case",
    "id",
    "text_original",
    "text_semantic_verified",
    "sentiment_label",
    "label_notes",
    "adjudication_reason"
]

review = flagged[columns].copy()

review = review.rename(columns={
    "sentiment_label": "previous_label",
    "label_notes": "previous_label_notes"
})

review["final_label"] = ""
review["adjudication_decision"] = ""
review["adjudication_notes"] = ""
review["reviewer"] = ""
review["reviewed_at"] = ""

with pd.ExcelWriter(
    ADJUDICATION_PATH,
    engine="openpyxl"
) as writer:

    review.to_excel(
        writer,
        sheet_name="ADJUDICATION_4",
        index=False
    )

    ws = writer.sheets["ADJUDICATION_4"]
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = ws.dimensions

    headers = {
        cell.value: cell.column_letter
        for cell in ws[1]
    }

    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(wrap_text=True)

    for col in ws.columns:
        name = str(col[0].value)
        letter = col[0].column_letter

        ws.column_dimensions[letter].width = (
            80 if name.startswith("text_") else 30
        )

        for cell in col[1:]:
            if isinstance(cell.value, str):
                cell.data_type = "s"
            if name == "id":
                cell.number_format = "@"

    label_validation = DataValidation(
        type="list",
        formula1='"POSITIF,NEGATIF,NETRAL"',
        allow_blank=True
    )
    ws.add_data_validation(label_validation)
    label_validation.add(
        f"{headers['final_label']}2:"
        f"{headers['final_label']}5"
    )

    decision_validation = DataValidation(
        type="list",
        formula1='"KEEP,CHANGE,UNRESOLVED"',
        allow_blank=True
    )
    ws.add_data_validation(decision_validation)
    decision_validation.add(
        f"{headers['adjudication_decision']}2:"
        f"{headers['adjudication_decision']}5"
    )

print("=" * 60)
print("LABELING 06 — ADJUDICATION PREPARATION")
print("=" * 60)
print("Flagged tweets :", len(review))
print("Source records :", len(audit))
print("Workbook       :", ADJUDICATION_PATH)
print("Auto relabeling: OFF")
print("ADJUDICATION PREPARATION: PASS")

display(
    review[
        ["case", "id", "previous_label",
         "adjudication_reason"]
    ]
)


LABELING 06 — ADJUDICATION PREPARATION
Flagged tweets : 4
Source records : 30
Workbook       : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\pilot\pilot_adjudication_4.xlsx
Auto relabeling: OFF
ADJUDICATION PREPARATION: PASS


,case,id,previous_label,adjudication_reason
6,lombok_2018,1028865436794810368,NEGATIF,Penolakan atau kritik terhadap pemerintah?
15,palu_2018,1046304416356130816,POSITIF,Pertanyaan atau dukungan eksplisit?
17,palu_2018,1047236117546774528,POSITIF,Judul berita atau dukungan penulis?
24,sumatra_2025,1996474933816529181,POSITIF,Laporan desakan pihak lain atau sikap penulis?


In [7]:
# LABELING 07 — PILOT ADJUDICATION RECONCILIATION
# 30 TWEETS | 4 MANUAL DECISIONS
# OFFLINE ONLY | IMMUTABLE SOURCE
# ============================================================

from pathlib import Path
import pandas as pd
import json

PILOT_DIR = OUTPUT_DIR / "pilot"

AUDIT_PATH = PILOT_DIR / "pilot_validation_audit.csv"
ADJ_PATH = PILOT_DIR / "pilot_adjudication_4.xlsx"

FINAL_CSV = PILOT_DIR / "pilot_labeled_final_30.csv"
FINAL_JSON = PILOT_DIR / "pilot_adjudication_summary.json"

for path in [AUDIT_PATH, ADJ_PATH]:
    assert path.is_file(), f"Input tidak ditemukan: {path}"

for path in [FINAL_CSV, FINAL_JSON]:
    if path.exists():
        raise FileExistsError(
            f"Output sudah ada: {path}. Jangan overwrite."
        )

# A — Load audited pilot and manual decisions
pilot_audit = pd.read_csv(
    AUDIT_PATH,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

adj = pd.read_excel(
    ADJ_PATH,
    sheet_name="ADJUDICATION_4",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

assert len(pilot_audit) == 30
assert len(adj) == 4
assert pilot_audit["id"].is_unique
assert adj["id"].is_unique

# B — Normalize annotation fields
for col in [
    "previous_label",
    "final_label",
    "adjudication_decision"
]:
    adj[col] = (
        adj[col].astype("string")
        .str.strip()
        .str.upper()
    )

VALID_LABELS = {"POSITIF", "NEGATIF", "NETRAL"}
VALID_DECISIONS = {"KEEP", "CHANGE"}

assert adj["final_label"].isin(VALID_LABELS).all()
assert adj["previous_label"].isin(VALID_LABELS).all()
assert adj["adjudication_decision"].isin(
    VALID_DECISIONS
).all()

for col in ["adjudication_notes", "reviewer", "reviewed_at"]:
    assert adj[col].astype("string").str.strip().ne("").all(), (
        f"Kolom belum lengkap: {col}"
    )

assert pd.to_datetime(
    adj["reviewed_at"], errors="coerce"
).notna().all(), "Tanggal adjudication tidak valid."

# C — Check all four flagged IDs
flagged_ids = set(
    pilot_audit.loc[
        pilot_audit["needs_adjudication"]
        .astype(str).str.lower().eq("true"),
        "id"
    ]
)

assert len(flagged_ids) == 4
assert set(adj["id"]) == flagged_ids

pilot_by_id = pilot_audit.set_index("id")
adj_by_id = adj.set_index("id")

# Verify immutable fields against the audited pilot.
for col in ["case", "text_original", "text_semantic_verified"]:
    assert (
        adj_by_id[col].astype(str).sort_index()
        == pilot_by_id.loc[
            adj_by_id.index, col
        ].astype(str).sort_index()
    ).all(), f"Mismatch in immutable column: {col}"

# Verify previous labels and decision consistency.
previous = (
    pilot_by_id.loc[
        adj_by_id.index, "sentiment_label"
    ].astype("string").str.upper()
)

assert (
    adj_by_id["previous_label"]
    .sort_index()
    .equals(previous.sort_index())
), "Previous labels do not match audited pilot."

is_same = (
    adj["previous_label"] == adj["final_label"]
)

assert (
    (adj["adjudication_decision"].eq("KEEP") & is_same)
    | (adj["adjudication_decision"].eq("CHANGE") & ~is_same)
).all(), "KEEP / CHANGE tidak konsisten."

# D — Apply validated decisions to a derived copy
pilot_final = pilot_audit.copy()

pilot_final["sentiment_label_before_adjudication"] = (
    pilot_final["sentiment_label"]
)

final_map = adj.set_index("id")["final_label"]

mask = pilot_final["id"].isin(final_map.index)

pilot_final.loc[mask, "sentiment_label"] = (
    pilot_final.loc[mask, "id"]
    .map(final_map)
    .to_numpy()
)

pilot_final["adjudication_status"] = "NOT_FLAGGED"

pilot_final.loc[
    mask, "adjudication_status"
] = "ADJUDICATED"

pilot_final["adjudication_notes"] = (
    pilot_final["id"]
    .map(adj.set_index("id")["adjudication_notes"])
    .fillna("")
)

# E — Validate final pilot
assert len(pilot_final) == 30
assert pilot_final["id"].is_unique
assert pilot_final["sentiment_label"].isin(
    VALID_LABELS
).all()

assert pilot_final["annotation_status"].eq("LABELED").all()

expected_distribution = {
    "POSITIF": 8,
    "NEGATIF": 2,
    "NETRAL": 20
}

distribution = pilot_final[
    "sentiment_label"
].value_counts().to_dict()

assert distribution == expected_distribution, (
    f"Unexpected label distribution: {distribution}"
)

assert pilot_final["adjudication_status"].eq(
    "ADJUDICATED"
).sum() == 4

# F — Save outputs
pilot_final.to_csv(
    FINAL_CSV,
    index=False,
    encoding="utf-8-sig"
)

summary = {
    "total_pilot": 30,
    "adjudicated": 4,
    "changed": int(
        adj["adjudication_decision"].eq("CHANGE").sum()
    ),
    "kept": int(
        adj["adjudication_decision"].eq("KEEP").sum()
    ),
    "unresolved": 0,
    "final_distribution": distribution,
    "status": "PILOT_ADJUDICATION_PASS"
}

FINAL_JSON.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("=" * 65)
print("LABELING 07 — PILOT ADJUDICATION RECONCILIATION")
print("=" * 65)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nFinal distribution by case:")
display(
    pilot_final.groupby(
        ["case", "sentiment_label"]
    ).size().unstack(fill_value=0)
)

print("\nOutput CSV :", FINAL_CSV)
print("Output JSON:", FINAL_JSON)
print("\nLABELING 07 RECONCILIATION: PASS")


LABELING 07 — PILOT ADJUDICATION RECONCILIATION
total_pilot: 30
adjudicated: 4
changed: 4
kept: 0
unresolved: 0
final_distribution: {'NETRAL': 20, 'POSITIF': 8, 'NEGATIF': 2}
status: PILOT_ADJUDICATION_PASS

Final distribution by case:


sentiment_label,NEGATIF,NETRAL,POSITIF
case,,,
lombok_2018,1,6,3
palu_2018,1,9,0
sumatra_2025,0,5,5



Output CSV : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\pilot\pilot_labeled_final_30.csv
Output JSON: C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\pilot\pilot_adjudication_summary.json

LABELING 07 RECONCILIATION: PASS


In [8]:
# LABELING 08 — CODEBOOK CALIBRATION v1.1
# BASED ON 30 PILOT TWEETS AND 4 ADJUDICATIONS
# OFFLINE ONLY | NO AUTOMATIC LABELING
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

PILOT_FINAL = (
    OUTPUT_DIR / "pilot" / "pilot_labeled_final_30.csv"
)

PILOT_SUMMARY = (
    OUTPUT_DIR / "pilot" / "pilot_adjudication_summary.json"
)

CODEBOOK_V1 = (
    CODEBOOK_DIR / "sentiment_codebook_v1.md"
)

CODEBOOK_V11 = (
    CODEBOOK_DIR / "sentiment_codebook_v1.1.md"
)

CALIBRATION_JSON = (
    CODEBOOK_DIR / "codebook_calibration_v1.1.json"
)

# ------------------------------------------------------------
# A. VALIDATE PILOT
# ------------------------------------------------------------

for path in [
    PILOT_FINAL,
    PILOT_SUMMARY,
    CODEBOOK_V1
]:
    assert path.is_file(), f"Missing input: {path}"

for path in [CODEBOOK_V11, CALIBRATION_JSON]:
    if path.exists():
        raise FileExistsError(
            f"Output sudah tersedia: {path}"
        )

pilot_final = pd.read_csv(
    PILOT_FINAL,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

pilot_summary = json.loads(
    PILOT_SUMMARY.read_text(encoding="utf-8")
)

assert len(pilot_final) == 30
assert pilot_final["id"].is_unique

assert pilot_summary["status"] == (
    "PILOT_ADJUDICATION_PASS"
)

assert pilot_final["sentiment_label"].isin({
    "POSITIF", "NEGATIF", "NETRAL"
}).all()

assert pilot_final["sentiment_label"].value_counts().to_dict() == {
    "NETRAL": 20,
    "POSITIF": 8,
    "NEGATIF": 2
}

assert pilot_final["adjudication_status"].eq(
    "ADJUDICATED"
).sum() == 4

# ------------------------------------------------------------
# B. WRITE CALIBRATED CODEBOOK
# ------------------------------------------------------------

codebook_v11 = """# Sentiment Labeling Codebook v1.1

Version: 1.1
Status: CALIBRATED_FOR_FULL_ANNOTATION
Source: Pilot 30 tweet, 4 keputusan adjudication

## 1. Tujuan

Mengklasifikasikan sikap penulis tweet terhadap
penetapan status bencana nasional untuk:
- Lombok 2018
- Palu 2018
- Sumatera 2025

Unit analisis utama adalah tweet dengan ID unik.

## 2. Target Sentimen

Objek penilaian adalah kebijakan penetapan
status bencana nasional.

Bukan sentimen umum terhadap:
- Bencana alam
- Korban bencana
- Bantuan kemanusiaan
- Pemerintah secara umum
- Partai politik atau tokoh politik

Kecuali pernyataan tersebut secara jelas
berkaitan dengan sikap terhadap penetapan status.

## 3. Kelas Sentimen

### POSITIF

Penulis secara jelas mendukung, menyetujui,
mendesak, atau menuntut penetapan status
bencana nasional.

Contoh ilustratif:
"Pemerintah harus segera menetapkan status
bencana nasional untuk Palu."

### NEGATIF

Penulis secara jelas menolak penetapan,
tidak menyetujui peningkatan status,
atau menyatakan status nasional tidak diperlukan.

Contoh ilustratif:
"Status bencana nasional tidak diperlukan."

### NETRAL

Penulis tidak memperlihatkan sikap
yang jelas terhadap penetapan status.

Termasuk:
- Laporan berita tanpa endorsement
- Kutipan pendapat orang lain
- Pertanyaan informatif
- Penjelasan pemerintah
- Informasi korban dan bantuan
- Kritik umum yang tidak menjelaskan sikap
  terhadap penetapan status

Contoh ilustratif:
"BNPB menjelaskan alasan status belum ditetapkan."

## 4. Aturan Keputusan Terstandardisasi

### R1 — Identifikasi objek
Pastikan opini mengarah ke penetapan status
bencana nasional, bukan sekadar peristiwa bencana.

### R2 — Identifikasi pemilik sikap
Pisahkan sikap penulis dari pernyataan pihak
ketiga yang diberitakan atau dikutip.

### R3 — Berita bukan otomatis dukungan
Judul seperti "PKS desak pemerintah menetapkan
status bencana nasional" diberi NETRAL apabila
hanya menyampaikan berita tanpa endorsement.

### R4 — Pertanyaan bukan otomatis dukungan
Pertanyaan tentang alasan atau proses penetapan
diberi NETRAL jika tidak menunjukkan tuntutan
atau penolakan yang jelas.

### R5 — Kritik pemerintah perlu konteks
Kritik terhadap pemerintah tidak otomatis POSITIF
atau NEGATIF. Tentukan apakah kritik tersebut
mendukung atau menolak penetapan.

### R6 — Kutipan dan repost
Kutipan dukungan atau penolakan pihak lain
tidak otomatis menjadi sikap penulis.
Cari komentar persetujuan atau penolakan yang jelas.

### R7 — Sindiran
Gunakan sikap tersirat hanya apabila maknanya
cukup jelas dari konteks lengkap. Jika tidak,
tandai untuk adjudication.

### R8 — Sikap campuran
Jika terdapat dukungan dan penolakan sekaligus
tanpa sikap dominan yang jelas, tandai AMBIGUOUS.

### R9 — Negasi
Perhatikan kata seperti tidak, belum, bukan,
tak, jangan, dan tidak perlu.
Kata "belum" saja bukan bukti penolakan.

### R10 — Konflik beberapa peristiwa
Jika tweet membahas beberapa bencana,
identifikasi sikap terhadap peristiwa target.
Apabila target dan sikap tidak jelas,
tandai AMBIGUOUS.

## 5. Alur Anotasi

1. Baca text_semantic_verified.
2. Periksa text_original jika konteks kurang jelas.
3. Identifikasi target penetapan.
4. Identifikasi pemilik sikap.
5. Tentukan POSITIF, NEGATIF, atau NETRAL.
6. Tulis alasan keputusan pada label_notes.
7. Untuk kasus tidak pasti, gunakan status AMBIGUOUS.
8. Selesaikan adjudication sebelum data final dikunci.

AMBIGUOUS adalah status review,
bukan kelas sentimen keempat.

## 6. Metadata Anotasi

Kolom yang perlu dicatat:
- sentiment_label
- annotation_status
- label_notes
- annotator
- labeled_at

Kolom sumber dan ID tidak boleh diubah.

## 7. Audit dan Konsistensi

- Satu ID memiliki satu label akhir.
- Tweet dengan teks identik perlu audit konsistensi.
- Keputusan reviewer harus terdokumentasi.
- Label modeling diturunkan berdasarkan ID.
- Jangan menggunakan prediksi model untuk
  menggantikan label manual.

## 8. Riwayat Kalibrasi

v1.0: aturan awal sebelum pilot.
v1.1: diperjelas setelah audit 30 tweet
dan adjudication terhadap 4 tweet ambigu.

Keempat keputusan adjudication mengubah
label menjadi NETRAL berdasarkan pemeriksaan
terhadap atribusi sikap dan konteks.

Distribusi pilot final:
- POSITIF: 8
- NEGATIF: 2
- NETRAL: 20

Distribusi pilot tidak digunakan untuk
mengestimasi proporsi sentimen keseluruhan.

## 9. Batasan

Pilot ini belum mengukur inter-annotator agreement.
Cohen's Kappa hanya dapat dihitung secara sah
jika ada anotasi independen dari dua anotator
terhadap item yang sama.

Codebook ini dapat digunakan untuk anotasi penuh,
dengan audit dan adjudication untuk kasus sulit.
"""

# ------------------------------------------------------------
# C. SAVE & VALIDATE
# ------------------------------------------------------------

CODEBOOK_V11.write_text(
    codebook_v11,
    encoding="utf-8"
)

saved = CODEBOOK_V11.read_text(encoding="utf-8")

required = [
    "POSITIF",
    "NEGATIF",
    "NETRAL",
    "AMBIGUOUS",
    "R2 — Identifikasi pemilik sikap",
    "R3 — Berita bukan otomatis dukungan",
    "R4 — Pertanyaan bukan otomatis dukungan",
    "CALIBRATED_FOR_FULL_ANNOTATION"
]

assert all(term in saved for term in required)

calibration = {
    "version": "1.1",
    "previous_version": "1.0",
    "pilot_size": 30,
    "adjudicated": 4,
    "label_changes": 4,
    "unresolved": 0,
    "final_distribution": {
        "POSITIF": 8,
        "NEGATIF": 2,
        "NETRAL": 20
    },
    "status": "CALIBRATED_FOR_FULL_ANNOTATION",
    "codebook_sha256": hashlib.sha256(
        saved.encode("utf-8")
    ).hexdigest()
}

CALIBRATION_JSON.write_text(
    json.dumps(
        calibration,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=" * 65)
print("LABELING 08 — CODEBOOK CALIBRATION")
print("=" * 65)
print("Previous version : 1.0")
print("Current version  : 1.1")
print("Pilot tweets     :", len(pilot_final))
print("Adjudicated      : 4")
print("Unresolved       : 0")
print("Codebook         :", CODEBOOK_V11)
print("Manifest         :", CALIBRATION_JSON)
print("CODEBOOK CALIBRATION: PASS")


LABELING 08 — CODEBOOK CALIBRATION
Previous version : 1.0
Current version  : 1.1
Pilot tweets     : 30
Adjudicated      : 4
Unresolved       : 0
Codebook         : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\codebook\sentiment_codebook_v1.1.md
Manifest         : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\codebook\codebook_calibration_v1.1.json
CODEBOOK CALIBRATION: PASS


In [9]:
# LABELING 09 — FULL ANNOTATION WORKBOOK PREPARATION
# 381 TWEETS | 30 PILOT LABELED | 351 PENDING
# OFFLINE ONLY | NO AUTO LABELING
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

# ------------------------------------------------------------
# A. PATHS
# ------------------------------------------------------------

PILOT_FINAL_PATH = (
    OUTPUT_DIR / "pilot" / "pilot_labeled_final_30.csv"
)

CODEBOOK_PATH = (
    CODEBOOK_DIR / "sentiment_codebook_v1.1.md"
)

CALIBRATION_PATH = (
    CODEBOOK_DIR / "codebook_calibration_v1.1.json"
)

FULL_DIR = OUTPUT_DIR / "full_annotation"
FULL_DIR.mkdir(parents=True, exist_ok=True)

FULL_WORKBOOK = (
    FULL_DIR / "full_annotation_381.xlsx"
)

FULL_MANIFEST = (
    FULL_DIR / "full_annotation_manifest.json"
)

for path in [
    PILOT_FINAL_PATH,
    CODEBOOK_PATH,
    CALIBRATION_PATH
]:
    assert path.is_file(), f"Missing input: {path}"

for path in [FULL_WORKBOOK, FULL_MANIFEST]:
    if path.exists():
        raise FileExistsError(
            f"Output sudah ada: {path}"
        )

# ------------------------------------------------------------
# B. LOAD & VERIFY CODEBOOK
# ------------------------------------------------------------

calibration = json.loads(
    CALIBRATION_PATH.read_text(encoding="utf-8")
)

codebook_text = CODEBOOK_PATH.read_text(
    encoding="utf-8"
)

codebook_hash = hashlib.sha256(
    codebook_text.encode("utf-8")
).hexdigest()

assert calibration["version"] == "1.1"
assert calibration["status"] == (
    "CALIBRATED_FOR_FULL_ANNOTATION"
)

assert (
    calibration["codebook_sha256"] == codebook_hash
), "Codebook hash mismatch."

# ------------------------------------------------------------
# C. LOAD ALL_381 AND FINAL PILOT
# ------------------------------------------------------------

full = df_all.copy()
full["id"] = full["id"].astype("string").str.strip()

pilot_final = pd.read_csv(
    PILOT_FINAL_PATH,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

pilot_final["id"] = (
    pilot_final["id"].astype("string").str.strip()
)

assert len(full) == 381
assert len(pilot_final) == 30

assert full["id"].is_unique
assert pilot_final["id"].is_unique

assert set(pilot_final["id"]).issubset(
    set(full["id"])
)

# ------------------------------------------------------------
# D. VALIDATE PILOT SOURCE INTEGRITY
# ------------------------------------------------------------

immutable_columns = [
    "case",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "model_selection_status",
    "final_text_hash"
]

full_idx = full.set_index("id")
pilot_idx = pilot_final.set_index("id")

for col in immutable_columns:
    expected = (
        full_idx.loc[pilot_idx.index, col]
        .astype("string")
    )

    actual = pilot_idx[col].astype("string")

    assert actual.sort_index().equals(
        expected.sort_index()
    ), f"Pilot source mismatch: {col}"

VALID_LABELS = {"POSITIF", "NEGATIF", "NETRAL"}

pilot_final["sentiment_label"] = (
    pilot_final["sentiment_label"]
    .astype("string").str.strip().str.upper()
)

assert pilot_final["sentiment_label"].isin(
    VALID_LABELS
).all()

assert pilot_final["sentiment_label"].value_counts().to_dict() == {
    "NETRAL": 20,
    "POSITIF": 8,
    "NEGATIF": 2
}

# ------------------------------------------------------------
# E. APPLY PILOT LABELS BY ID
# ------------------------------------------------------------

full["sentiment_label"] = ""
full["annotation_status"] = "PENDING"
full["label_notes"] = ""
full["annotator"] = ""
full["labeled_at"] = ""
full["annotation_source"] = "FULL_MANUAL"

pilot_indexed = pilot_final.set_index("id")

pilot_ids = set(pilot_final["id"])

pilot_mask = full["id"].isin(pilot_ids)

for target_col, source_col in [
    ("sentiment_label", "sentiment_label"),
    ("label_notes", "label_notes"),
    ("annotator", "annotator"),
    ("labeled_at", "labeled_at"),
]:
    full.loc[pilot_mask, target_col] = (
        full.loc[pilot_mask, "id"]
        .map(pilot_indexed[source_col])
        .to_numpy()
    )

full.loc[
    pilot_mask, "annotation_status"
] = "LABELED"

full.loc[
    pilot_mask, "annotation_source"
] = "PILOT_ADJUDICATED"

# ------------------------------------------------------------
# F. VERIFY INITIAL COUNTS
# ------------------------------------------------------------

assert len(full) == 381
assert full["id"].is_unique

assert full["annotation_status"].eq(
    "LABELED"
).sum() == 30

assert full["annotation_status"].eq(
    "PENDING"
).sum() == 351

assert full["annotation_source"].eq(
    "PILOT_ADJUDICATED"
).sum() == 30

assert full.loc[
    ~pilot_mask, "sentiment_label"
].eq("").all()

assert (
    full.loc[pilot_mask, "sentiment_label"]
    .value_counts().to_dict()
) == {
    "NETRAL": 20,
    "POSITIF": 8,
    "NEGATIF": 2
}

# ------------------------------------------------------------
# G. WORKBOOK DATA
# ------------------------------------------------------------

annotation_cols = [
    "case",
    "id",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "model_selection_status",
    "duplicate_group_id",
    "duplicate_group_size",
    "long_text_review_status",
    "final_text_hash",
    "sentiment_label",
    "annotation_status",
    "label_notes",
    "annotator",
    "labeled_at",
    "annotation_source"
]

annotation_df = full[annotation_cols].copy()

readme = pd.DataFrame([
    {
        "Topic": "Target sentiment",
        "Instruction": (
            "Sikap penulis terhadap penetapan "
            "status bencana nasional."
        )
    },
    {
        "Topic": "POSITIF",
        "Instruction": "Mendukung penetapan status."
    },
    {
        "Topic": "NEGATIF",
        "Instruction": "Menolak atau menilai status tidak diperlukan."
    },
    {
        "Topic": "NETRAL",
        "Instruction": "Tidak ada sikap penulis yang jelas."
    },
    {
        "Topic": "PILOT_ADJUDICATED",
        "Instruction": (
            "30 tweet sudah memiliki label pilot final. "
            "Jangan diubah tanpa audit baru."
        )
    },
    {
        "Topic": "FULL_MANUAL",
        "Instruction": (
            "351 tweet menunggu anotasi manual "
            "berdasarkan codebook v1.1."
        )
    },
    {
        "Topic": "AMBIGUOUS",
        "Instruction": (
            "Gunakan sebagai annotation_status, "
            "biarkan sentiment_label kosong, "
            "dan tulis alasannya."
        )
    },
    {
        "Topic": "Duplicate groups",
        "Instruction": (
            "Anotasi tetap berdasarkan ID. "
            "Konsistensi konten identik diaudit setelah selesai."
        )
    }
])

# ------------------------------------------------------------
# H. WRITE EXCEL
# ------------------------------------------------------------

with pd.ExcelWriter(
    FULL_WORKBOOK,
    engine="openpyxl"
) as writer:

    readme.to_excel(
        writer,
        sheet_name="README",
        index=False
    )

    annotation_df.to_excel(
        writer,
        sheet_name="ALL_381",
        index=False
    )

    ws = writer.sheets["ALL_381"]
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = ws.dimensions

    header_map = {
        cell.value: cell.column_letter
        for cell in ws[1]
    }

    for sheet in writer.sheets.values():
        for cell in sheet[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(wrap_text=True)

    for col in ws.columns:
        header = str(col[0].value)
        letter = col[0].column_letter

        ws.column_dimensions[letter].width = (
            75 if header.startswith("text_") else 25
        )

        for cell in col[1:]:
            if isinstance(cell.value, str):
                cell.data_type = "s"

            if header == "id":
                cell.number_format = "@"

    label_validation = DataValidation(
        type="list",
        formula1='"POSITIF,NEGATIF,NETRAL"',
        allow_blank=True
    )
    ws.add_data_validation(label_validation)

    status_validation = DataValidation(
        type="list",
        formula1='"PENDING,LABELED,AMBIGUOUS,ADJUDICATED"',
        allow_blank=False
    )
    ws.add_data_validation(status_validation)

    label_col = header_map["sentiment_label"]
    status_col = header_map["annotation_status"]

    label_validation.add(
        f"{label_col}2:{label_col}382"
    )
    status_validation.add(
        f"{status_col}2:{status_col}382"
    )

    # Bedakan baris pilot secara visual.
    source_col_index = annotation_cols.index(
        "annotation_source"
    ) + 1

    for row in range(2, ws.max_row + 1):
        source_value = ws.cell(
            row, source_col_index
        ).value

        if source_value == "PILOT_ADJUDICATED":
            for cell in ws[row]:
                cell.fill = PatternFill(
                    "solid",
                    fgColor="E8F2FC"
                )

    ws.sheet_view.showGridLines = False

# ------------------------------------------------------------
# I. MANIFEST
# ------------------------------------------------------------

manifest_full = {
    "stage": "03_Labeling",
    "part": "Full Annotation Preparation",
    "codebook_version": "1.1",
    "codebook_sha256": codebook_hash,
    "total_tweets": 381,
    "pilot_labeled": 30,
    "pending_manual": 351,
    "model_candidates": 348,
    "pilot_distribution": {
        "POSITIF": 8,
        "NEGATIF": 2,
        "NETRAL": 20
    },
    "status": "READY_FOR_FULL_MANUAL_ANNOTATION"
}

FULL_MANIFEST.write_text(
    json.dumps(
        manifest_full,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=" * 65)
print("LABELING 09 — FULL ANNOTATION PREPARATION")
print("=" * 65)
print("Total tweets   :", len(full))
print("Pilot labeled  :", int(pilot_mask.sum()))
print("Pending manual :", int((~pilot_mask).sum()))
print("Codebook       : v1.1")
print("Workbook       :", FULL_WORKBOOK)
print("Manifest       :", FULL_MANIFEST)
print("\nLABELING 09 FULL PREPARATION: PASS")


LABELING 09 — FULL ANNOTATION PREPARATION
Total tweets   : 381
Pilot labeled  : 30
Pending manual : 351
Codebook       : v1.1
Workbook       : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\full_annotation\full_annotation_381.xlsx
Manifest       : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\full_annotation\full_annotation_manifest.json

LABELING 09 FULL PREPARATION: PASS


In [14]:
# LABELING 10-DIAG — ANNOTATION STATUS DIAGNOSTIC
# READ ONLY | NO SOURCE MODIFICATION
# ============================================================

import pandas as pd

workbook_path = (
    ROOT / "03_Labeling" / "outputs"
    / "full_annotation" / "full_annotation_381.xlsx"
)

check = pd.read_excel(
    workbook_path,
    sheet_name="ALL_381",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

check["id"] = check["id"].astype("string").str.strip()

for col in ["sentiment_label", "annotation_status"]:
    check[col] = (
        check[col].astype("string")
        .str.strip().str.upper()
    )

VALID_LABELS = {"POSITIF", "NEGATIF", "NETRAL"}
COMPLETED = {"LABELED", "ADJUDICATED"}

# Simpan nomor baris Excel untuk navigasi.
check["excel_row"] = range(2, len(check) + 2)

# Kondisi 1: label terisi, tetapi status belum selesai.
problem_status = check.loc[
    check["sentiment_label"].isin(VALID_LABELS)
    & ~check["annotation_status"].isin(COMPLETED)
].copy()

# Kondisi 2: status selesai, tetapi label tidak valid.
problem_label = check.loc[
    check["annotation_status"].isin(COMPLETED)
    & ~check["sentiment_label"].isin(VALID_LABELS)
].copy()

# Kondisi 3: label/status tidak dikenali.
valid_statuses = {
    "PENDING", "LABELED", "AMBIGUOUS", "ADJUDICATED"
}

invalid_values = check.loc[
    ~check["annotation_status"].isin(valid_statuses)
    | (
        check["sentiment_label"].ne("")
        & ~check["sentiment_label"].isin(VALID_LABELS)
    )
].copy()

print("=" * 65)
print("LABELING 10-DIAG — STATUS DIAGNOSTIC")
print("=" * 65)

print("Total tweet                    :", len(check))
print("Label valid terisi             :",
      int(check["sentiment_label"].isin(VALID_LABELS).sum()))
print("Status LABELED / ADJUDICATED   :",
      int(check["annotation_status"].isin(COMPLETED).sum()))
print("PENDING                        :",
      int(check["annotation_status"].eq("PENDING").sum()))
print("AMBIGUOUS                      :",
      int(check["annotation_status"].eq("AMBIGUOUS").sum()))

print("\nLabel terisi tetapi status belum selesai:",
      len(problem_status))

print("Status selesai tetapi label invalid:",
      len(problem_label))

print("Nilai label/status tidak dikenali:",
      len(invalid_values))

columns_to_show = [
    "excel_row",
    "case",
    "id",
    "sentiment_label",
    "annotation_status",
    "label_notes",
    "annotator",
    "labeled_at"
]

print("\nA. BARIS DENGAN STATUS TIDAK KONSISTEN")
display(problem_status[columns_to_show])

print("\nB. BARIS DENGAN LABEL TIDAK VALID")
display(problem_label[columns_to_show])

print("\nC. NILAI YANG TIDAK DIKENALI")
display(invalid_values[columns_to_show])

print("\nD. DISTRIBUSI KOMBINASI STATUS DAN LABEL")
display(
    check.groupby(
        ["annotation_status", "sentiment_label"]
    ).size().reset_index(name="count")
)

print("\nDIAGNOSTIC COMPLETE — WORKBOOK UNCHANGED")


LABELING 10-DIAG — STATUS DIAGNOSTIC
Total tweet                    : 381
Label valid terisi             : 130
Status LABELED / ADJUDICATED   : 130
PENDING                        : 251
AMBIGUOUS                      : 0

Label terisi tetapi status belum selesai: 0
Status selesai tetapi label invalid: 0
Nilai label/status tidak dikenali: 0

A. BARIS DENGAN STATUS TIDAK KONSISTEN


,excel_row,case,id,sentiment_label,annotation_status,label_notes,annotator,labeled_at



B. BARIS DENGAN LABEL TIDAK VALID


,excel_row,case,id,sentiment_label,annotation_status,label_notes,annotator,labeled_at



C. NILAI YANG TIDAK DIKENALI


,excel_row,case,id,sentiment_label,annotation_status,label_notes,annotator,labeled_at



D. DISTRIBUSI KOMBINASI STATUS DAN LABEL


,annotation_status,sentiment_label,count
0,LABELED,NEGATIF,4
1,LABELED,NETRAL,94
2,LABELED,POSITIF,32
3,PENDING,,251



DIAGNOSTIC COMPLETE — WORKBOOK UNCHANGED


In [21]:
# LABELING 10 — FULL ANNOTATION PROGRESS AUDIT
# 381 TWEETS | PILOT LOCK | BATCH A 100
# OFFLINE ONLY | NO AUTOMATIC LABELING
# ============================================================

from pathlib import Path
import pandas as pd
import json

# ------------------------------------------------------------
# A. CONFIGURATION
# ------------------------------------------------------------

FULL_DIR = OUTPUT_DIR / "full_annotation"
WORKBOOK = FULL_DIR / "full_annotation_381.xlsx"

SOURCE_CSV = INPUT_DIR / "labeling_all_381.csv"
PILOT_CSV = (
    OUTPUT_DIR / "pilot" / "pilot_labeled_final_30.csv"
)

AUDIT_DIR = FULL_DIR / "progress_audits"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = AUDIT_DIR / "annotation_progress.json"
DETAILS_PATH = AUDIT_DIR / "annotation_progress_details.csv"
BATCH_A_PATH = AUDIT_DIR / "batch_a_100_ids.csv"

for path in [WORKBOOK, SOURCE_CSV, PILOT_CSV]:
    assert path.is_file(), f"Missing file: {path}"

# ------------------------------------------------------------
# B. LOAD WORKBOOK AND SOURCE
# ------------------------------------------------------------

df = pd.read_excel(
    WORKBOOK,
    sheet_name="ALL_381",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

source = pd.read_csv(
    SOURCE_CSV,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

pilot = pd.read_csv(
    PILOT_CSV,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

required_columns = [
    "case",
    "id",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "model_selection_status",
    "final_text_hash",
    "sentiment_label",
    "annotation_status",
    "label_notes",
    "annotator",
    "labeled_at",
    "annotation_source"
]

missing = set(required_columns) - set(df.columns)
assert not missing, f"Missing columns: {missing}"

for frame in [df, source, pilot]:
    frame["id"] = frame["id"].astype("string").str.strip()

assert len(df) == len(source) == 381
assert len(pilot) == 30

assert df["id"].is_unique
assert source["id"].is_unique
assert pilot["id"].is_unique

assert set(df["id"]) == set(source["id"])

# ------------------------------------------------------------
# C. SOURCE INTEGRITY
# ------------------------------------------------------------

immutable = [
    "case",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "model_selection_status",
    "final_text_hash"
]

workbook_by_id = df.set_index("id")
source_by_id = source.set_index("id")

for column in immutable:
    expected = source_by_id[column].astype("string")
    actual = workbook_by_id[column].astype("string")

    assert actual.sort_index().equals(
        expected.sort_index()
    ), f"SOURCE MISMATCH: {column}"

print("SOURCE INTEGRITY: PASS")

# ------------------------------------------------------------
# D. NORMALIZE ANNOTATION FIELDS FOR AUDIT
# ------------------------------------------------------------

for column in [
    "sentiment_label",
    "annotation_status",
    "annotation_source"
]:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

VALID_LABELS = {"POSITIF", "NEGATIF", "NETRAL"}

VALID_STATUSES = {
    "PENDING",
    "LABELED",
    "AMBIGUOUS",
    "ADJUDICATED"
}

assert df["annotation_status"].isin(
    VALID_STATUSES
).all(), "Invalid annotation status."

assert df["annotation_source"].isin({
    "PILOT_ADJUDICATED",
    "FULL_MANUAL"
}).all(), "Invalid annotation source."

# ------------------------------------------------------------
# E. VALIDATE LABEL / STATUS COMBINATIONS
# ------------------------------------------------------------

completed_mask = df["annotation_status"].isin({
    "LABELED",
    "ADJUDICATED"
})

pending_mask = df["annotation_status"].eq("PENDING")
ambiguous_mask = df["annotation_status"].eq("AMBIGUOUS")

assert df.loc[
    completed_mask, "sentiment_label"
].isin(VALID_LABELS).all(), (
    "Ada LABELED/ADJUDICATED dengan label tidak valid."
)

assert df.loc[
    pending_mask | ambiguous_mask,
    "sentiment_label"
].eq("").all(), (
    "PENDING/AMBIGUOUS tidak boleh memiliki label final."
)

for col in ["label_notes", "annotator", "labeled_at"]:
    values = df[col].astype("string").str.strip()

    assert values.loc[completed_mask].ne("").all(), (
        f"Kolom {col} kosong pada tweet selesai."
    )

assert df.loc[
    ambiguous_mask, "label_notes"
].astype("string").str.strip().ne("").all(), (
    "Tweet AMBIGUOUS wajib memiliki alasan."
)

# ------------------------------------------------------------
# F. LOCK 30 PILOT ANNOTATIONS
# ------------------------------------------------------------

pilot_by_id = pilot.set_index("id")

pilot_ids = set(pilot["id"])
pilot_mask = df["id"].isin(pilot_ids)

assert int(pilot_mask.sum()) == 30

for column in [
    "sentiment_label",
    "label_notes",
    "annotator",
    "labeled_at"
]:
    actual = (
        df.set_index("id")
        .loc[pilot_by_id.index, column]
        .astype("string")
    )

    expected = pilot_by_id[column].astype("string")

    assert actual.sort_index().equals(
        expected.sort_index()
    ), f"PILOT LOCK FAILED: {column}"

assert df.loc[
    pilot_mask, "annotation_source"
].eq("PILOT_ADJUDICATED").all()

assert df.loc[
    pilot_mask, "annotation_status"
].eq("LABELED").all()

assert df.loc[
    ~pilot_mask, "annotation_source"
].eq("FULL_MANUAL").all()

print("PILOT LOCK: PASS")

# ------------------------------------------------------------
# G. CREATE STABLE BATCH A ROSTER
# ------------------------------------------------------------

# Excel row = posisi baris pada sheet ALL_381.
# Berlaku untuk urutan workbook saat roster dibuat.
df["excel_row_at_preparation"] = range(2, len(df) + 2)

if not BATCH_A_PATH.exists():

    # Buat roster hanya saat kondisi masih awal.
    nonpilot_completed = (
        completed_mask & ~pilot_mask
    ).sum()

    assert int(nonpilot_completed) == 0, (
        "Batch roster belum dibuat, tetapi anotasi "
        "non-pilot sudah dimulai. Periksa manual."
    )

    candidates = df.loc[
        ~pilot_mask & pending_mask
    ].copy()

    assert len(candidates) == 351

    batch_a = candidates.head(100)[
        [
            "case",
            "id",
            "excel_row_at_preparation"
        ]
    ].copy()

    batch_a.to_csv(
        BATCH_A_PATH,
        index=False,
        encoding="utf-8-sig"
    )

else:
    batch_a = pd.read_csv(
        BATCH_A_PATH,
        dtype={"id": "string"},
        encoding="utf-8-sig"
    )

    batch_a["id"] = (
        batch_a["id"].astype("string").str.strip()
    )

assert len(batch_a) == 100
assert batch_a["id"].is_unique
assert set(batch_a["id"]).issubset(set(df["id"]))
assert not set(batch_a["id"]) & pilot_ids

# ------------------------------------------------------------
# H. PROGRESS REPORT
# ------------------------------------------------------------

completed = int(completed_mask.sum())
pending = int(pending_mask.sum())
ambiguous = int(ambiguous_mask.sum())

batch_a_rows = df.loc[
    df["id"].isin(batch_a["id"])
]

batch_a_completed = int(
    batch_a_rows["annotation_status"]
    .isin({"LABELED", "ADJUDICATED"})
    .sum()
)

batch_a_pending = int(
    batch_a_rows["annotation_status"]
    .eq("PENDING")
    .sum()
)

batch_a_ambiguous = int(
    batch_a_rows["annotation_status"]
    .eq("AMBIGUOUS")
    .sum()
)

assert len(batch_a_rows) == 100
assert completed + pending + ambiguous == 381

label_distribution = (
    df.loc[completed, "sentiment_label"]
    if False else
    df.loc[completed_mask, "sentiment_label"]
).value_counts().to_dict()

case_progress = (
    df.groupby(["case", "annotation_status"])
    .size()
    .unstack(fill_value=0)
)

summary = {
    "total_tweets": 381,
    "pilot_locked": 30,
    "completed": completed,
    "pending": pending,
    "ambiguous": ambiguous,
    "label_distribution": label_distribution,
    "batch_a_target": 100,
    "batch_a_completed": batch_a_completed,
    "batch_a_pending": batch_a_pending,
    "batch_a_ambiguous": batch_a_ambiguous,
    "source_integrity": "PASS",
    "pilot_integrity": "PASS",
    "status": (
        "ALL_LABELS_COMPLETED"
        if completed == 381
        else "ANNOTATION_IN_PROGRESS"
    )
}

# Progress reports adalah snapshot terbaru.
# Roster Batch A TIDAK ditimpa.
DETAIL_COLS = [
    "case",
    "id",
    "annotation_status",
    "sentiment_label",
    "annotation_source"
]

df[DETAIL_COLS].to_csv(
    DETAILS_PATH,
    index=False,
    encoding="utf-8-sig"
)

SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=" * 65)
print("LABELING 10 — FULL ANNOTATION PROGRESS AUDIT")
print("=" * 65)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nProgress by case:")
display(case_progress)

print("\nBatch A — first 10 IDs:")
display(batch_a.head(10))

print("\nAudit summary :", SUMMARY_PATH)
print("Audit details :", DETAILS_PATH)
print("Batch A roster:", BATCH_A_PATH)

print("\nLABELING 10 PROGRESS AUDIT: PASS")
print("Source workbook: NOT MODIFIED")
print("Pilot labels: LOCKED")
print("Automatic labeling: OFF")


SOURCE INTEGRITY: PASS
PILOT LOCK: PASS
LABELING 10 — FULL ANNOTATION PROGRESS AUDIT
total_tweets: 381
pilot_locked: 30
completed: 130
pending: 251
ambiguous: 0
label_distribution: {'NETRAL': 94, 'POSITIF': 32, 'NEGATIF': 4}
batch_a_target: 100
batch_a_completed: 100
batch_a_pending: 0
batch_a_ambiguous: 0
source_integrity: PASS
pilot_integrity: PASS
status: ANNOTATION_IN_PROGRESS

Progress by case:


annotation_status,LABELED,PENDING
case,,
lombok_2018,46,68
palu_2018,53,65
sumatra_2025,31,118



Batch A — first 10 IDs:


,case,id,excel_row_at_preparation
0,lombok_2018,1026427713525927938,4
1,lombok_2018,1026424653298855937,5
2,lombok_2018,1026421529775493121,6
3,lombok_2018,1026411069466533888,7
4,lombok_2018,1026403261195071489,8
5,lombok_2018,1026402242776117249,9
6,lombok_2018,1026398913371074562,10
7,lombok_2018,1026390946693271553,11
8,lombok_2018,1026372308284264448,12
9,palu_2018,1046068799915642880,13



Audit summary : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\full_annotation\progress_audits\annotation_progress.json
Audit details : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\full_annotation\progress_audits\annotation_progress_details.csv
Batch A roster: C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\full_annotation\progress_audits\batch_a_100_ids.csv

LABELING 10 PROGRESS AUDIT: PASS
Source workbook: NOT MODIFIED
Pilot labels: LOCKED
Automatic labeling: OFF


In [37]:
# LABELING 11 — DUPLICATE LABEL ADJUDICATION PREPARATION
# READ ONLY MASTER | NO LABEL MODIFICATION
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
LABEL_DIR = ROOT / "03_Labeling"

MASTER_PATH = (
    LABEL_DIR / "outputs" / "full_annotation"
    / "full_annotation_381.xlsx"
)

OUTPUT_DIR = LABEL_DIR / "outputs" / "final_labeling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_PATH = OUTPUT_DIR / "duplicate_adjudication_4.xlsx"
SNAPSHOT_PATH = OUTPUT_DIR / "duplicate_conflict_snapshot.csv"
MANIFEST_PATH = OUTPUT_DIR / "duplicate_adjudication_manifest.json"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def csv_read(path):
    return pd.read_csv(
        path,
        dtype={"id": "string"},
        keep_default_na=False,
        encoding="utf-8-sig"
    )

for path in [REVIEW_PATH, SNAPSHOT_PATH, MANIFEST_PATH]:
    if path.exists():
        raise FileExistsError(
            f"Output sudah ada: {path}. Jangan overwrite."
        )

assert MASTER_PATH.is_file()

df381 = pd.read_excel(
    MASTER_PATH,
    sheet_name="ALL_381",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

df381["id"] = df381["id"].astype("string").str.strip()

required = [
    "case", "id", "text_original",
    "text_semantic_verified", "final_text_hash",
    "duplicate_group_id",
    "model_selection_status",
    "sentiment_label", "annotation_status",
    "label_notes", "annotator", "labeled_at"
]

assert set(required).issubset(df381.columns)
assert len(df381) == 381
assert df381["id"].is_unique

for col in ["sentiment_label", "annotation_status"]:
    df381[col] = (
        df381[col].astype("string")
        .str.strip().str.upper()
    )

assert df381["sentiment_label"].isin(
    {"POSITIF", "NEGATIF", "NETRAL"}
).all()

assert df381["annotation_status"].eq("LABELED").all()

for col in ["label_notes", "annotator", "labeled_at"]:
    assert df381[col].astype("string").str.strip().ne("").all()


# ============================================================
# DETECT ACTUAL DUPLICATE GROUPS
# FIX: ONLY GROUPS CONTAINING MORE THAN ONE TWEET
# ============================================================

# Semua tweet memiliki duplicate_group_id,
# termasuk tweet yang hanya muncul satu kali.
# Oleh sebab itu, identifikasi duplikat harus
# menggunakan jumlah tweet unik per grup.

df381["duplicate_group_id"] = (
    df381["duplicate_group_id"]
    .astype("string")
    .str.strip()
)

# A. Hitung jumlah tweet unik pada setiap grup.
all_group_sizes = (
    df381.groupby("duplicate_group_id")["id"]
    .nunique()
)

# B. Grup duplikat sebenarnya: lebih dari satu ID.
actual_duplicate_ids = all_group_sizes.loc[
    all_group_sizes > 1
].index

duplicate_rows = df381.loc[
    df381["duplicate_group_id"].isin(
        actual_duplicate_ids
    )
].copy()

group_sizes = (
    duplicate_rows.groupby("duplicate_group_id")["id"]
    .nunique()
)

# C. Deteksi konflik label hanya dalam grup duplikat.
conflict_counts = (
    duplicate_rows.groupby(
        "duplicate_group_id"
    )["sentiment_label"].nunique()
)

conflict_groups = conflict_counts.loc[
    conflict_counts > 1
].index.tolist()

conflicts = duplicate_rows.loc[
    duplicate_rows["duplicate_group_id"].isin(
        conflict_groups
    )
].copy()

conflicts = conflicts.sort_values(
    ["duplicate_group_id", "id"]
).reset_index(drop=True)

print("=" * 65)
print("ACTUAL DUPLICATE GROUP AUDIT")
print("=" * 65)
print("All unique group IDs :", len(all_group_sizes))
print("Actual duplicate groups:", len(group_sizes))
print("Duplicate rows       :", len(duplicate_rows))
print("Conflicting groups   :", len(conflict_groups))
print("Conflicting rows     :", len(conflicts))

# D. Assertions sesuai audit Cleaning.
assert len(df381) == 381
assert len(all_group_sizes) == 348

assert len(group_sizes) == 21, (
    "Expected 21 actual duplicate groups."
)

assert len(duplicate_rows) == 54, (
    "Expected 54 rows in duplicate groups."
)

assert len(conflict_groups) == 2, (
    "Expected 2 conflicting groups."
)

assert len(conflicts) == 4, (
    "Expected 4 conflicting tweets."
)

# E. Pastikan teks hasil cleaning dalam
# setiap grup konflik benar-benar identik.
assert conflicts.groupby(
    "duplicate_group_id"
)["final_text_hash"].nunique().eq(1).all(), (
    "Conflicting group contains different text hashes."
)

print("\nConflict details:")

display(
    conflicts[
        [
            "case",
            "id",
            "duplicate_group_id",
            "sentiment_label",
            "model_selection_status"
        ]
    ]
)

print("\nDUPLICATE GROUP DETECTION: PASS")

snapshot_cols = [
    "case", "id", "duplicate_group_id",
    "text_original", "text_semantic_verified",
    "final_text_hash", "model_selection_status",
    "sentiment_label", "label_notes",
    "annotation_status", "annotator", "labeled_at"
]

snapshot = conflicts[snapshot_cols].copy()

snapshot.to_csv(
    SNAPSHOT_PATH,
    index=False,
    encoding="utf-8-sig"
)

review = snapshot.rename(columns={
    "sentiment_label": "previous_label"
}).copy()

review["final_label"] = ""
review["adjudication_decision"] = ""
review["adjudication_notes"] = ""
review["reviewer"] = ""
review["reviewed_at"] = ""

readme = pd.DataFrame([
    {
        "Field": "Goal",
        "Instruction": (
            "Selesaikan 2 konflik label di antara "
            "4 tweet dari grup duplikat."
        )
    },
    {
        "Field": "final_label",
        "Instruction": (
            "Pilih POSITIF, NEGATIF, atau NETRAL "
            "berdasarkan codebook v1.1."
        )
    },
    {
        "Field": "adjudication_decision",
        "Instruction": (
            "KEEP jika sama dengan previous_label; "
            "CHANGE jika berbeda."
        )
    },
    {
        "Field": "Group consistency",
        "Instruction": (
            "Tweet dalam satu grup harus memiliki "
            "final_label yang konsisten jika konteks sama."
        )
    },
    {
        "Field": "Review metadata",
        "Instruction": (
            "Isi adjudication_notes, reviewer, "
            "reviewed_at (YYYY-MM-DD)."
        )
    },
    {
        "Field": "Source",
        "Instruction": (
            "Jangan mengubah kolom selain 5 kolom "
            "adjudication yang disediakan."
        )
    }
])

with pd.ExcelWriter(
    REVIEW_PATH,
    engine="openpyxl"
) as writer:

    readme.to_excel(
        writer, sheet_name="README", index=False
    )
    review.to_excel(
        writer, sheet_name="ADJUDICATION_4", index=False
    )

    ws = writer.sheets["ADJUDICATION_4"]
    ws.freeze_panes = "D2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill(
            "solid", fgColor="1F4E78"
        )

    for col in ws.columns:
        field = str(col[0].value)
        letter = col[0].column_letter

        ws.column_dimensions[letter].width = (
            75 if field.startswith("text_")
            else 55 if field.endswith("_notes")
            else 26
        )

        for cell in col[1:]:
            cell.alignment = Alignment(
                vertical="top", wrap_text=True
            )
            if field == "id":
                cell.number_format = "@"

    for field in [
        "final_label", "adjudication_decision",
        "adjudication_notes", "reviewer", "reviewed_at"
    ]:
        col_index = review.columns.get_loc(field) + 1
        for row_idx in range(2, ws.max_row + 1):
            ws.cell(row_idx, col_index).fill = PatternFill(
                "solid", fgColor="FFF2CC"
            )

    validations = [
        ("final_label", '"POSITIF,NEGATIF,NETRAL"'),
        ("adjudication_decision", '"KEEP,CHANGE"')
    ]

    for field, options in validations:
        dv = DataValidation(
            type="list",
            formula1=options,
            allow_blank=False
        )
        ws.add_data_validation(dv)
        letter = ws.cell(
            1, review.columns.get_loc(field) + 1
        ).column_letter
        dv.add(f"{letter}2:{letter}{ws.max_row}")

manifest = {
    "stage": "03_Labeling",
    "part": "Duplicate Label Adjudication",
    "master_sha256": sha256(MASTER_PATH),
    "snapshot_sha256": sha256(SNAPSHOT_PATH),
    "duplicate_groups": int(len(group_sizes)),
    "conflict_groups": 2,
    "conflict_rows": 4,
    "status": "AWAITING_MANUAL_ADJUDICATION"
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\nConflicting labels:")
display(
    review[
        [
            "duplicate_group_id", "id",
            "previous_label", "model_selection_status"
        ]
    ]
)

print("\nWorkbook :", REVIEW_PATH)
print("Snapshot :", SNAPSHOT_PATH)
print("Manifest :", MANIFEST_PATH)
print("\nLABELING 11 DUPLICATE PREPARATION: PASS")


ACTUAL DUPLICATE GROUP AUDIT
All unique group IDs : 348
Actual duplicate groups: 21
Duplicate rows       : 54
Conflicting groups   : 2
Conflicting rows     : 4

Conflict details:


,case,id,duplicate_group_id,sentiment_label,model_selection_status
0,palu_2018,1046695376504291328,c47204a7bfba4814,POSITIF,SELECTED_REPRESENTATIVE
1,palu_2018,1046727803301838848,c47204a7bfba4814,NETRAL,GROUP_MEMBER_NOT_SELECTED
2,palu_2018,1046688954014031872,dda331d2c37d509d,NETRAL,SELECTED_REPRESENTATIVE
3,palu_2018,1046689466507620352,dda331d2c37d509d,POSITIF,GROUP_MEMBER_NOT_SELECTED



DUPLICATE GROUP DETECTION: PASS

Conflicting labels:


,duplicate_group_id,id,previous_label,model_selection_status
0,c47204a7bfba4814,1046695376504291328,POSITIF,SELECTED_REPRESENTATIVE
1,c47204a7bfba4814,1046727803301838848,NETRAL,GROUP_MEMBER_NOT_SELECTED
2,dda331d2c37d509d,1046688954014031872,NETRAL,SELECTED_REPRESENTATIVE
3,dda331d2c37d509d,1046689466507620352,POSITIF,GROUP_MEMBER_NOT_SELECTED



Workbook : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\final_labeling\duplicate_adjudication_4.xlsx
Snapshot : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\final_labeling\duplicate_conflict_snapshot.csv
Manifest : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\final_labeling\duplicate_adjudication_manifest.json

LABELING 11 DUPLICATE PREPARATION: PASS


In [47]:
# LABELING 12-DIAG — FINAL LABEL VALIDATION
# READ ONLY | NO DATA MODIFICATION
# ============================================================

import pandas as pd
from pathlib import Path

review_path = (
    Path(r"C:\Penelitian Sentimen Bencana Nasional")
    / "03_Labeling"
    / "outputs"
    / "final_labeling"
    / "duplicate_adjudication_4.xlsx"
)

df_review = pd.read_excel(
    review_path,
    sheet_name="ADJUDICATION_4",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

valid_labels = {"POSITIF", "NEGATIF", "NETRAL"}

df_review["final_label_normalized"] = (
    df_review["final_label"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_review["label_valid"] = (
    df_review["final_label_normalized"]
    .isin(valid_labels)
)

print("=" * 65)
print("LABELING 12-DIAG — FINAL LABEL VALIDATION")
print("=" * 65)

display(
    df_review[
        [
            "id",
            "duplicate_group_id",
            "previous_label",
            "final_label",
            "final_label_normalized",
            "adjudication_decision",
            "label_valid"
        ]
    ]
)

invalid = df_review.loc[
    ~df_review["label_valid"]
]

print("\nTotal reviewed :", len(df_review))
print("Valid labels   :", int(df_review["label_valid"].sum()))
print("Invalid labels :", len(invalid))

if not invalid.empty:
    print("\nROWS THAT REQUIRE MANUAL COMPLETION:")
    display(
        invalid[
            ["id", "previous_label", "final_label"]
        ]
    )
else:
    print("\nFINAL LABEL VALIDATION: PASS")


LABELING 12-DIAG — FINAL LABEL VALIDATION


,id,duplicate_group_id,previous_label,final_label,final_label_normalized,adjudication_decision,label_valid
0,1046695376504291328,c47204a7bfba4814,POSITIF,POSITIF,POSITIF,KEEP,True
1,1046727803301838848,c47204a7bfba4814,NETRAL,POSITIF,POSITIF,CHANGE,True
2,1046688954014031872,dda331d2c37d509d,NETRAL,NETRAL,NETRAL,KEEP,True
3,1046689466507620352,dda331d2c37d509d,POSITIF,NETRAL,NETRAL,CHANGE,True



Total reviewed : 4
Valid labels   : 4
Invalid labels : 0

FINAL LABEL VALIDATION: PASS


In [49]:
# LABELING 12 — DUPLICATE ADJUDICATION RECONCILIATION
# FIXED: KeyError 'id'
# 381 TWEETS | 2 CONFLICT GROUPS | 4 REVIEWED ROWS
# DERIVED OUTPUT ONLY | MASTER XLSX UNCHANGED
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

# ------------------------------------------------------------
# A. PATH CONFIGURATION
# ------------------------------------------------------------

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

LABEL_DIR = ROOT / "03_Labeling"

FINAL_DIR = (
    LABEL_DIR / "outputs" / "final_labeling"
)

MASTER_PATH = (
    LABEL_DIR / "outputs"
    / "full_annotation"
    / "full_annotation_381.xlsx"
)

REVIEW_PATH = (
    FINAL_DIR / "duplicate_adjudication_4.xlsx"
)

SNAPSHOT_PATH = (
    FINAL_DIR / "duplicate_conflict_snapshot.csv"
)

MANIFEST_PATH = (
    FINAL_DIR / "duplicate_adjudication_manifest.json"
)

FINAL_381_PATH = (
    FINAL_DIR / "labeled_final_381.csv"
)

CHANGELOG_PATH = (
    FINAL_DIR / "duplicate_label_changelog.csv"
)

RECON_PATH = (
    FINAL_DIR / "duplicate_reconciliation.json"
)

# ------------------------------------------------------------
# B. UTILITIES
# ------------------------------------------------------------

def sha256_file(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


def read_csv_ids(path):
    df = pd.read_csv(
        path,
        dtype={"id": "string"},
        keep_default_na=False,
        encoding="utf-8-sig"
    )

    df["id"] = (
        df["id"]
        .astype("string")
        .str.strip()
    )

    return df


def normalized_series(frame, column):
    values = frame[column].copy()

    if column in {"labeled_at", "reviewed_at"}:
        raw = values.astype("string").str.strip()

        parsed = pd.to_datetime(
            raw,
            errors="coerce",
            format="mixed"
        )

        assert parsed.notna().all(), (
            f"Invalid or missing date: {column}"
        )

        return (
            parsed.dt.strftime("%Y-%m-%d")
            .sort_index()
        )

    return (
        values.astype("string")
        .str.strip()
        .fillna("")
        .sort_index()
    )



# ------------------------------------------------------------
# C. CHECK ALL REQUIRED INPUTS
# ------------------------------------------------------------

required_inputs = [
    MASTER_PATH,
    REVIEW_PATH,
    SNAPSHOT_PATH,
    MANIFEST_PATH
]

for path in required_inputs:
    assert path.is_file(), (
        f"Required input missing: {path}"
    )

# No silent overwrite of previous reconciliation.
output_paths = [
    FINAL_381_PATH,
    CHANGELOG_PATH,
    RECON_PATH
]

existing_outputs = [
    str(path)
    for path in output_paths
    if path.exists()
]

assert not existing_outputs, (
    "Output reconciliation sudah tersedia. "
    "Periksa dahulu sebelum menjalankan ulang:\n"
    + "\n".join(existing_outputs)
)

# ------------------------------------------------------------
# D. LOAD AND VERIFY MANIFEST
# ------------------------------------------------------------

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert manifest["status"] == (
    "AWAITING_MANUAL_ADJUDICATION"
)

assert manifest["conflict_groups"] == 2
assert manifest["conflict_rows"] == 4

assert sha256_file(MASTER_PATH) == (
    manifest["master_sha256"]
), (
    "MASTER WORKBOOK CHANGED since preparation. "
    "Stop reconciliation."
)

assert sha256_file(SNAPSHOT_PATH) == (
    manifest["snapshot_sha256"]
), (
    "CONFLICT SNAPSHOT CHANGED. "
    "Stop reconciliation."
)

master_hash_before = sha256_file(MASTER_PATH)

print("MANIFEST INTEGRITY: PASS")
print("MASTER INTEGRITY: PASS")
print("SNAPSHOT INTEGRITY: PASS")

# ------------------------------------------------------------
# E. LOAD SNAPSHOT AND MANUAL REVIEW
# ------------------------------------------------------------

snapshot = read_csv_ids(SNAPSHOT_PATH)

review = pd.read_excel(
    REVIEW_PATH,
    sheet_name="ADJUDICATION_4",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

review["id"] = (
    review["id"]
    .astype("string")
    .str.strip()
)

assert len(snapshot) == 4
assert len(review) == 4

assert snapshot["id"].is_unique
assert review["id"].is_unique

assert set(review["id"]) == set(
    snapshot["id"]
), (
    "Tweet IDs do not match snapshot."
)

assert review["id"].str.strip().ne("").all()

print("REVIEW ROW COUNT: PASS")
print("REVIEW ID INTEGRITY: PASS")

# ------------------------------------------------------------
# F. VALIDATE IMMUTABLE REVIEW COLUMNS
# FIX FOR PREVIOUS KeyError 'id'
# ------------------------------------------------------------

review_idx = review.set_index("id")
snapshot_idx = snapshot.set_index("id")

# ID is already used as index.
# Therefore, exclude it from column iteration.
immutable_columns = [
    col
    for col in snapshot.columns
    if col != "id"
]

for column in immutable_columns:

    review_column = (
        "previous_label"
        if column == "sentiment_label"
        else column
    )

    assert review_column in review_idx.columns, (
        f"Missing review column: {review_column}"
    )

    assert column in snapshot_idx.columns, (
        f"Missing snapshot column: {column}"
    )

    actual = normalized_series(
        review_idx,
        review_column
    )

    expected = normalized_series(
        snapshot_idx,
        column
    )

    assert actual.equals(expected), (
        f"IMMUTABLE FIELD CHANGED: {column}"
    )

print("IMMUTABLE FIELDS: PASS")

# ------------------------------------------------------------
# G. VALIDATE ADJUDICATION DECISIONS
# ------------------------------------------------------------

VALID_LABELS = {
    "POSITIF",
    "NEGATIF",
    "NETRAL"
}

VALID_DECISIONS = {
    "KEEP",
    "CHANGE"
}

for column in [
    "previous_label",
    "final_label",
    "adjudication_decision"
]:
    review[column] = (
        review[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

assert review["previous_label"].isin(
    VALID_LABELS
).all()

assert review["final_label"].isin(
    VALID_LABELS
).all(), (
    "Invalid or missing final_label."
)

assert review["adjudication_decision"].isin(
    VALID_DECISIONS
).all(), (
    "Invalid or missing adjudication_decision."
)

same_label = (
    review["previous_label"]
    == review["final_label"]
)

decision_consistent = (
    (
        review["adjudication_decision"].eq("KEEP")
        & same_label
    )
    |
    (
        review["adjudication_decision"].eq("CHANGE")
        & ~same_label
    )
)

assert decision_consistent.all(), (
    "KEEP / CHANGE inconsistent with labels."
)

for column in [
    "adjudication_notes",
    "reviewer",
    "reviewed_at"
]:
    assert review[column].astype(
        "string"
    ).str.strip().ne("").all(), (
        f"Missing adjudication metadata: {column}"
    )

review_dates = pd.to_datetime(
    review["reviewed_at"],
    errors="coerce"
)

assert review_dates.notna().all(), (
    "Invalid adjudication dates."
)

# ------------------------------------------------------------
# H. VALIDATE GROUP CONSISTENCY
# ------------------------------------------------------------

assert review[
    "duplicate_group_id"
].nunique() == 2

assert review.groupby(
    "duplicate_group_id"
)["id"].nunique().eq(2).all(), (
    "Expected two tweets in each conflict group."
)

assert review.groupby(
    "duplicate_group_id"
)["final_label"].nunique().eq(1).all(), (
    "Duplicate group final labels are inconsistent."
)

assert review.groupby(
    "duplicate_group_id"
)["final_text_hash"].nunique().eq(1).all(), (
    "Duplicate group text hashes are inconsistent."
)

print("ADJUDICATION DECISIONS: PASS")
print("DUPLICATE GROUP CONSISTENCY: PASS")

# ------------------------------------------------------------
# I. LOAD ORIGINAL MASTER
# ------------------------------------------------------------

original = pd.read_excel(
    MASTER_PATH,
    sheet_name="ALL_381",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

original["id"] = (
    original["id"]
    .astype("string")
    .str.strip()
)

assert len(original) == 381
assert original["id"].is_unique

assert set(review["id"]).issubset(
    set(original["id"])
)

original["sentiment_label"] = (
    original["sentiment_label"]
    .astype("string")
    .str.strip()
    .str.upper()
)

assert original["sentiment_label"].isin(
    VALID_LABELS
).all()

assert original["annotation_status"].astype(
    "string"
).str.strip().str.upper().eq(
    "LABELED"
).all()

for column in [
    "label_notes",
    "annotator",
    "labeled_at"
]:
    assert original[column].astype(
        "string"
    ).str.strip().ne("").all(), (
        f"Missing original metadata: {column}"
    )

# Verify previous labels in review match master.
master_idx = original.set_index("id")
review_idx = review.set_index("id")

assert (
    master_idx.loc[
        review_idx.index,
        "sentiment_label"
    ].astype("string").sort_index()
    .equals(
        review_idx[
            "previous_label"
        ].astype("string").sort_index()
    )
), "Review previous_label differs from master."

print("ORIGINAL MASTER VALIDATION: PASS")

# ------------------------------------------------------------
# J. APPLY DECISIONS TO DERIVED COPY ONLY
# ------------------------------------------------------------

final = original.copy(deep=True)

final["sentiment_label_original"] = (
    final["sentiment_label"]
)

final["adjudication_applied"] = False
final["adjudication_notes"] = ""

review_by_id = review.set_index("id")

affected_mask = final["id"].isin(
    set(review["id"])
)

final.loc[
    affected_mask,
    "sentiment_label"
] = (
    final.loc[
        affected_mask,
        "id"
    ].map(
        review_by_id["final_label"]
    ).to_numpy()
)

final.loc[
    affected_mask,
    "adjudication_applied"
] = True

final.loc[
    affected_mask,
    "adjudication_notes"
] = (
    final.loc[
        affected_mask,
        "id"
    ].map(
        review_by_id[
            "adjudication_notes"
        ]
    ).to_numpy()
)

assert len(final) == 381
assert final["id"].is_unique

assert final["sentiment_label"].isin(
    VALID_LABELS
).all()

# ------------------------------------------------------------
# K. VERIFY EXACT NUMBER OF LABEL CHANGES
# ------------------------------------------------------------

actual_changes = (
    final["sentiment_label"]
    != final["sentiment_label_original"]
)

changed_count = int(
    actual_changes.sum()
)

expected_changes = int(
    review[
        "adjudication_decision"
    ].eq("CHANGE").sum()
)

assert changed_count == expected_changes, (
    "Unexpected number of label changes."
)

assert final[
    "adjudication_applied"
].sum() == 4

# No changes outside the four reviewed IDs.
assert final.loc[
    ~affected_mask,
    "sentiment_label"
].equals(
    final.loc[
        ~affected_mask,
        "sentiment_label_original"
    ]
)

print("APPLIED DECISIONS: PASS")
print("ACTUAL LABEL CHANGES:", changed_count)

# ------------------------------------------------------------
# L. VALIDATE ALL 21 ACTUAL DUPLICATE GROUPS
# ------------------------------------------------------------

group_counts = final.groupby(
    "duplicate_group_id"
)["id"].nunique()

actual_duplicate_group_ids = (
    group_counts.loc[
        group_counts > 1
    ].index
)

actual_duplicate_rows = final.loc[
    final["duplicate_group_id"].isin(
        actual_duplicate_group_ids
    )
]

assert len(actual_duplicate_group_ids) == 21, (
    "Expected 21 actual duplicate groups."
)

assert len(actual_duplicate_rows) == 54, (
    "Expected 54 rows in duplicate groups."
)

remaining_conflicts = actual_duplicate_rows.groupby(
    "duplicate_group_id"
)["sentiment_label"].nunique()

assert remaining_conflicts.eq(1).all(), (
    "Some duplicate groups still have "
    "inconsistent sentiment labels."
)

assert actual_duplicate_rows.groupby(
    "duplicate_group_id"
)["final_text_hash"].nunique().eq(1).all(), (
    "Text hashes inconsistent within duplicate groups."
)

print("ALL 21 DUPLICATE GROUPS: PASS")
print("REMAINING CONFLICTS: 0")

# ------------------------------------------------------------
# M. VERIFY ORIGINAL FIELDS UNCHANGED
# FIX: EXCLUDE ID FROM COLUMN COMPARISON
# ------------------------------------------------------------

before_idx = original.set_index("id")
after_idx = final.set_index("id")

# Validasi ID secara terpisah.
assert original["id"].is_unique
assert final["id"].is_unique

assert set(original["id"]) == set(final["id"]), (
    "Original and final tweet IDs differ."
)

# ID sudah menjadi index, sehingga tidak boleh
# diakses lagi sebagai kolom.
immutable_master_columns = [
    col for col in original.columns
    if col not in {"id", "sentiment_label"}
]

for column in immutable_master_columns:

    actual = normalized_series(
        after_idx,
        column
    )

    expected = normalized_series(
        before_idx,
        column
    )

    assert actual.equals(expected), (
        f"Unexpected change in original field: {column}"
    )

# Catatan manual lama wajib dipertahankan.
assert normalized_series(
    before_idx, "label_notes"
).equals(
    normalized_series(
        after_idx, "label_notes"
    )
), "Original annotation notes were modified."

# Label di luar 4 ID adjudication tidak boleh berubah.
reviewed_ids = set(review["id"])

before_labels = (
    before_idx.loc[
        ~before_idx.index.isin(reviewed_ids),
        "sentiment_label"
    ]
    .astype("string")
    .sort_index()
)

after_labels = (
    after_idx.loc[
        ~after_idx.index.isin(reviewed_ids),
        "sentiment_label"
    ]
    .astype("string")
    .sort_index()
)

assert before_labels.equals(after_labels), (
    "Labels outside adjudication were modified."
)

print("SOURCE AND ORIGINAL METADATA: UNCHANGED")
print("NON-REVIEWED LABELS: UNCHANGED")
print("MASTER IMMUTABILITY VALIDATION: PASS")

# ------------------------------------------------------------
# N. PREPARE CHANGELOG AND REPORT
# ------------------------------------------------------------

changelog = review[
    [
        "case",
        "id",
        "duplicate_group_id",
        "previous_label",
        "final_label",
        "adjudication_decision",
        "adjudication_notes",
        "reviewer",
        "reviewed_at"
    ]
].copy()

reconciliation = {
    "stage": "03_Labeling",
    "part": "Duplicate Adjudication Reconciliation",
    "total_tweets": 381,
    "reviewed_rows": 4,
    "reviewed_groups": 2,
    "changed_labels": changed_count,
    "kept_labels": 4 - changed_count,
    "total_duplicate_groups": 21,
    "total_duplicate_rows": 54,
    "conflicting_groups_remaining": 0,
    "master_modified": False,
    "status": "DUPLICATE_RECONCILIATION_PASS"
}

# ------------------------------------------------------------
# O. SAVE OUTPUT FILES
# ------------------------------------------------------------

final.to_csv(
    FINAL_381_PATH,
    index=False,
    encoding="utf-8-sig"
)

changelog.to_csv(
    CHANGELOG_PATH,
    index=False,
    encoding="utf-8-sig"
)

RECON_PATH.write_text(
    json.dumps(
        reconciliation,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# Recheck master was not modified.
assert sha256_file(MASTER_PATH) == master_hash_before, (
    "Master workbook changed unexpectedly."
)

# Read final CSV back to verify persistence.
final_check = read_csv_ids(FINAL_381_PATH)

assert len(final_check) == 381
assert final_check["id"].is_unique
assert final_check["sentiment_label"].isin(
    VALID_LABELS
).all()

# ------------------------------------------------------------
# P. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("LABELING 12 — DUPLICATE RECONCILIATION")
print("=" * 65)

for key, value in reconciliation.items():
    print(f"{key}: {value}")

print("\nFinal label distribution:")
display(
    final["sentiment_label"]
    .value_counts()
)

print("\nAdjudication changelog:")
display(
    changelog[
        [
            "id",
            "previous_label",
            "final_label",
            "adjudication_decision"
        ]
    ]
)

print("\nOutputs:")
print("Final 381 :", FINAL_381_PATH)
print("Changelog :", CHANGELOG_PATH)
print("Report    :", RECON_PATH)

print("\nLABELING 12 RECONCILIATION: PASS")
print("MASTER WORKBOOK: UNCHANGED")


MANIFEST INTEGRITY: PASS
MASTER INTEGRITY: PASS
SNAPSHOT INTEGRITY: PASS
REVIEW ROW COUNT: PASS
REVIEW ID INTEGRITY: PASS
IMMUTABLE FIELDS: PASS
ADJUDICATION DECISIONS: PASS
DUPLICATE GROUP CONSISTENCY: PASS
ORIGINAL MASTER VALIDATION: PASS
APPLIED DECISIONS: PASS
ACTUAL LABEL CHANGES: 2
ALL 21 DUPLICATE GROUPS: PASS
REMAINING CONFLICTS: 0
SOURCE AND ORIGINAL METADATA: UNCHANGED
NON-REVIEWED LABELS: UNCHANGED
MASTER IMMUTABILITY VALIDATION: PASS

LABELING 12 — DUPLICATE RECONCILIATION
stage: 03_Labeling
part: Duplicate Adjudication Reconciliation
total_tweets: 381
reviewed_rows: 4
reviewed_groups: 2
changed_labels: 2
kept_labels: 2
total_duplicate_groups: 21
total_duplicate_rows: 54
conflicting_groups_remaining: 0
master_modified: False
status: DUPLICATE_RECONCILIATION_PASS

Final label distribution:


sentiment_label
NETRAL     253
POSITIF    113
NEGATIF     15
Name: count, dtype: Int64


Adjudication changelog:


,id,previous_label,final_label,adjudication_decision
0,1046695376504291328,POSITIF,POSITIF,KEEP
1,1046727803301838848,NETRAL,POSITIF,CHANGE
2,1046688954014031872,NETRAL,NETRAL,KEEP
3,1046689466507620352,POSITIF,NETRAL,CHANGE



Outputs:
Final 381 : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\final_labeling\labeled_final_381.csv
Changelog : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\final_labeling\duplicate_label_changelog.csv
Report    : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\final_labeling\duplicate_reconciliation.json

LABELING 12 RECONCILIATION: PASS
MASTER WORKBOOK: UNCHANGED


In [50]:
# LABELING 13 — FINAL AUDIT & MODELING HANDOFF
# 381 LABELED TWEETS -> 348 UNIQUE MODEL CANDIDATES
#
# READ ONLY:
# - full_annotation_381.xlsx
# - labeled_final_381.csv
# - duplicate_label_changelog.csv
# - duplicate_reconciliation.json
# - labeling_candidates_348.csv
#
# NEW OUTPUT ONLY:
# - modeling_labeled_348.csv
# - final_labeling_audit.json
#
# NO MASTER MODIFICATION | NO AUTOMATIC RELABELING
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import hashlib
import json

# ------------------------------------------------------------
# A. PATH CONFIGURATION
# ------------------------------------------------------------

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

LABEL_DIR = ROOT / "03_Labeling"

INPUT_DIR = LABEL_DIR / "inputs"

FINAL_DIR = (
    LABEL_DIR / "outputs" / "final_labeling"
)

MASTER_PATH = (
    LABEL_DIR / "outputs"
    / "full_annotation"
    / "full_annotation_381.xlsx"
)

FINAL_381_PATH = (
    FINAL_DIR / "labeled_final_381.csv"
)

CHANGELOG_PATH = (
    FINAL_DIR / "duplicate_label_changelog.csv"
)

RECON_PATH = (
    FINAL_DIR / "duplicate_reconciliation.json"
)

CANDIDATES_PATH = (
    INPUT_DIR / "labeling_candidates_348.csv"
)

HANDOFF_PATH = (
    FINAL_DIR / "modeling_labeled_348.csv"
)

AUDIT_PATH = (
    FINAL_DIR / "final_labeling_audit.json"
)

# ------------------------------------------------------------
# B. CONFIGURATION
# ------------------------------------------------------------

EXPECTED_TOTAL = 381
EXPECTED_CANDIDATES = 348
EXPECTED_NONSELECTED = 33
EXPECTED_UNIQUE_GROUPS = 348
EXPECTED_DUPLICATE_GROUPS = 21
EXPECTED_DUPLICATE_ROWS = 54
EXPECTED_ADJUDICATED_ROWS = 4

VALID_LABELS = {
    "POSITIF",
    "NEGATIF",
    "NETRAL"
}

SOURCE_COLUMNS = [
    "case",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "final_text_hash",
    "duplicate_group_id",
    "model_selection_status"
]

ANNOTATION_COLUMNS = [
    "sentiment_label",
    "sentiment_label_original",
    "annotation_status",
    "label_notes",
    "annotator",
    "labeled_at",
    "annotation_source",
    "adjudication_applied",
    "adjudication_notes"
]

# ------------------------------------------------------------
# C. HELPER FUNCTIONS
# ------------------------------------------------------------

def sha256_file(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


def read_csv_with_ids(path):
    frame = pd.read_csv(
        path,
        dtype={"id": "string"},
        keep_default_na=False,
        encoding="utf-8-sig"
    )

    assert "id" in frame.columns, (
        f"Missing id column: {path}"
    )

    frame["id"] = (
        frame["id"]
        .astype("string")
        .str.strip()
    )

    return frame


def normalized_text(series):
    return (
        series
        .astype("string")
        .fillna("")
        .str.strip()
    )


def normalized_label(series):
    return (
        normalized_text(series)
        .str.upper()
    )


def ensure_unique_ids(frame, name):
    assert "id" in frame.columns, (
        f"{name}: missing id"
    )

    assert frame["id"].notna().all(), (
        f"{name}: null IDs found"
    )

    assert frame["id"].str.strip().ne("").all(), (
        f"{name}: empty IDs found"
    )

    assert frame["id"].is_unique, (
        f"{name}: duplicate IDs found"
    )


def compare_fields_by_id(
    left,
    right,
    columns,
    left_name,
    right_name
):
    left_idx = left.set_index("id")
    right_idx = right.set_index("id")

    assert set(left_idx.index) == set(right_idx.index), (
        f"ID mismatch: {left_name} vs {right_name}"
    )

    for column in columns:
        assert column in left_idx.columns, (
            f"{left_name}: missing {column}"
        )
        assert column in right_idx.columns, (
            f"{right_name}: missing {column}"
        )

        a = normalized_text(
            left_idx[column]
        ).sort_index()

        b = normalized_text(
            right_idx[column]
        ).sort_index()

        assert a.equals(b), (
            f"Mismatch in {column}: "
            f"{left_name} vs {right_name}"
        )


def count_distribution(frame):
    return {
        str(key): int(value)
        for key, value in (
            frame["sentiment_label"]
            .value_counts()
            .items()
        )
    }


# ------------------------------------------------------------
# D. INPUT CHECKS
# ------------------------------------------------------------

print("=" * 70)
print("LABELING 13 — FINAL AUDIT & MODELING HANDOFF")
print("=" * 70)

required_inputs = [
    MASTER_PATH,
    FINAL_381_PATH,
    CHANGELOG_PATH,
    RECON_PATH,
    CANDIDATES_PATH
]

for path in required_inputs:
    assert path.is_file(), (
        f"Required input missing: {path}"
    )

assert not HANDOFF_PATH.exists(), (
    f"Handoff output already exists: {HANDOFF_PATH}"
)

assert not AUDIT_PATH.exists(), (
    f"Audit output already exists: {AUDIT_PATH}"
)

input_hashes_before = {
    str(path): sha256_file(path)
    for path in required_inputs
}

print("REQUIRED INPUTS: PASS")
print("NO OUTPUT OVERWRITE: PASS")

# ------------------------------------------------------------
# E. LOAD FILES
# ------------------------------------------------------------

master = pd.read_excel(
    MASTER_PATH,
    sheet_name="ALL_381",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

master["id"] = (
    master["id"]
    .astype("string")
    .str.strip()
)

final = read_csv_with_ids(FINAL_381_PATH)
candidates = read_csv_with_ids(CANDIDATES_PATH)
changelog = read_csv_with_ids(CHANGELOG_PATH)

reconciliation = json.loads(
    RECON_PATH.read_text(
        encoding="utf-8"
    )
)

# ------------------------------------------------------------
# F. BASIC VALIDATION
# ------------------------------------------------------------

assert len(master) == EXPECTED_TOTAL
assert len(final) == EXPECTED_TOTAL

assert len(candidates) == EXPECTED_CANDIDATES

assert len(changelog) == (
    EXPECTED_ADJUDICATED_ROWS
)

ensure_unique_ids(master, "MASTER")
ensure_unique_ids(final, "FINAL 381")
ensure_unique_ids(candidates, "CANDIDATES 348")
ensure_unique_ids(changelog, "CHANGELOG")

assert set(master["id"]) == set(final["id"])

assert set(candidates["id"]).issubset(
    set(final["id"])
)

assert set(changelog["id"]).issubset(
    set(final["id"])
)

print("MASTER ROWS:", len(master))
print("FINAL ROWS :", len(final))
print("CANDIDATES :", len(candidates))
print("CHANGELOG  :", len(changelog))
print("ID INTEGRITY: PASS")

# ------------------------------------------------------------
# G. LABEL AND ANNOTATION COMPLETENESS
# ------------------------------------------------------------

for frame in [master, final]:
    frame["sentiment_label"] = normalized_label(
        frame["sentiment_label"]
    )

    frame["annotation_status"] = normalized_label(
        frame["annotation_status"]
    )

    assert frame["sentiment_label"].isin(
        VALID_LABELS
    ).all(), "Invalid sentiment label found."

    assert frame["annotation_status"].eq(
        "LABELED"
    ).all(), "Incomplete annotation status."

    for column in [
        "label_notes",
        "annotator",
        "labeled_at"
    ]:
        assert column in frame.columns

        assert normalized_text(
            frame[column]
        ).ne("").all(), (
            f"Empty annotation metadata: {column}"
        )

assert normalized_label(
    final["sentiment_label_original"]
).isin(VALID_LABELS).all()

print("ALL 381 LABELS: VALID")
print("ANNOTATION NOTES: COMPLETE")
print("ANNOTATOR METADATA: COMPLETE")
print("LABELING STATUS: PASS")

# ------------------------------------------------------------
# H. MASTER-TO-FINAL SOURCE INTEGRITY
# ------------------------------------------------------------

compare_fields_by_id(
    master,
    final,
    SOURCE_COLUMNS,
    "MASTER",
    "FINAL"
)

# Original annotations must remain recoverable.
master_idx = master.set_index("id")
final_idx = final.set_index("id")

assert normalized_label(
    master_idx["sentiment_label"]
).sort_index().equals(
    normalized_label(
        final_idx["sentiment_label_original"]
    ).sort_index()
), (
    "Original sentiment labels differ from master."
)

# Compare original manual notes.
assert normalized_text(
    master_idx["label_notes"]
).sort_index().equals(
    normalized_text(
        final_idx["label_notes"]
    ).sort_index()
), (
    "Original annotation notes were changed."
)

# Check original annotation metadata.
for column in [
    "annotation_status",
    "annotation_source",
    "annotator"
]:
    assert normalized_text(
        master_idx[column]
    ).sort_index().equals(
        normalized_text(
            final_idx[column]
        ).sort_index()
    ), f"Original metadata changed: {column}"

# Dates may have equivalent Excel/CSV representations.
master_dates = pd.to_datetime(
    master_idx["labeled_at"].astype("string"),
    format="mixed",
    errors="coerce"
)

final_dates = pd.to_datetime(
    final_idx["labeled_at"].astype("string"),
    format="mixed",
    errors="coerce"
)

assert master_dates.notna().all()
assert final_dates.notna().all()

assert (
    master_dates.dt.strftime("%Y-%m-%d")
    .sort_index()
    .equals(
        final_dates.dt.strftime("%Y-%m-%d")
        .sort_index()
    )
), "Original labeling dates differ."

print("MASTER SOURCE INTEGRITY: PASS")
print("ORIGINAL ANNOTATIONS PRESERVED: PASS")

# ------------------------------------------------------------
# I. ADJUDICATION AUDIT
# ------------------------------------------------------------

assert reconciliation["status"] == (
    "DUPLICATE_RECONCILIATION_PASS"
)

assert reconciliation["total_tweets"] == 381
assert reconciliation["reviewed_rows"] == 4
assert reconciliation[
    "conflicting_groups_remaining"
] == 0

assert reconciliation[
    "master_modified"
] is False

changelog["previous_label"] = normalized_label(
    changelog["previous_label"]
)

changelog["final_label"] = normalized_label(
    changelog["final_label"]
)

changelog["adjudication_decision"] = (
    normalized_label(
        changelog["adjudication_decision"]
    )
)

assert changelog["previous_label"].isin(
    VALID_LABELS
).all()

assert changelog["final_label"].isin(
    VALID_LABELS
).all()

assert changelog[
    "adjudication_decision"
].isin({"KEEP", "CHANGE"}).all()

expected_changed_ids = set(
    changelog.loc[
        changelog["adjudication_decision"].eq(
            "CHANGE"
        ),
        "id"
    ]
)

actual_changed_ids = set(
    final.loc[
        normalized_label(
            final["sentiment_label"]
        ) != normalized_label(
            final["sentiment_label_original"]
        ),
        "id"
    ]
)

assert actual_changed_ids == expected_changed_ids, (
    "Final label changes do not match changelog."
)

assert len(actual_changed_ids) == int(
    reconciliation["changed_labels"]
)

assert set(changelog["id"]) == set(
    final.loc[
        final["adjudication_applied"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("true"),
        "id"
    ]
)

for column in [
    "adjudication_notes",
    "reviewer",
    "reviewed_at"
]:
    assert normalized_text(
        changelog[column]
    ).ne("").all()

# Validate each changelog decision against final.
final_lookup = final.set_index("id")

for row in changelog.itertuples(index=False):
    tweet_id = str(row.id)

    assert (
        str(final_lookup.at[
            tweet_id, "sentiment_label"
        ]).strip().upper()
        == row.final_label
    )

    assert (
        str(final_lookup.at[
            tweet_id, "sentiment_label_original"
        ]).strip().upper()
        == row.previous_label
    )

    expected_decision = (
        "KEEP"
        if row.previous_label == row.final_label
        else "CHANGE"
    )

    assert row.adjudication_decision == (
        expected_decision
    )

print("ADJUDICATION CHANGELOG: PASS")
print("ACTUAL LABEL CHANGES:", len(actual_changed_ids))

# ------------------------------------------------------------
# J. DUPLICATE GROUP AUDIT
# ------------------------------------------------------------

final["duplicate_group_id"] = normalized_text(
    final["duplicate_group_id"]
)

final["final_text_hash"] = normalized_text(
    final["final_text_hash"]
)

assert final["duplicate_group_id"].ne("").all()
assert final["final_text_hash"].ne("").all()

group_sizes = (
    final.groupby("duplicate_group_id")["id"]
    .nunique()
)

assert len(group_sizes) == EXPECTED_UNIQUE_GROUPS

actual_duplicate_ids = group_sizes.loc[
    group_sizes > 1
].index

duplicates = final.loc[
    final["duplicate_group_id"].isin(
        actual_duplicate_ids
    )
].copy()

assert len(actual_duplicate_ids) == (
    EXPECTED_DUPLICATE_GROUPS
)

assert len(duplicates) == EXPECTED_DUPLICATE_ROWS

assert duplicates.groupby(
    "duplicate_group_id"
)["sentiment_label"].nunique().eq(1).all(), (
    "Duplicate label conflicts remain."
)

assert duplicates.groupby(
    "duplicate_group_id"
)["final_text_hash"].nunique().eq(1).all(), (
    "Duplicate text hashes are inconsistent."
)

# Ensure a text hash does not belong to multiple groups.
assert final.groupby(
    "final_text_hash"
)["duplicate_group_id"].nunique().eq(1).all(), (
    "Same final text hash assigned "
    "to multiple duplicate groups."
)

assert final["final_text_hash"].nunique() == (
    EXPECTED_CANDIDATES
)

print("UNIQUE TEXT GROUPS:", len(group_sizes))
print("ACTUAL DUPLICATE GROUPS:", len(actual_duplicate_ids))
print("DUPLICATE ROWS:", len(duplicates))
print("CONFLICTING LABEL GROUPS: 0")
print("DUPLICATE AUDIT: PASS")

# ------------------------------------------------------------
# K. CANDIDATE SOURCE INTEGRITY
# ------------------------------------------------------------

# Compare candidate records with the final 381 dataset.
# Only IDs approved during cleaning may be selected.

selected_final = final.loc[
    final["id"].isin(
        set(candidates["id"])
    )
].copy()

assert len(selected_final) == EXPECTED_CANDIDATES

# Candidate CSV must retain its approved source data.
candidate_idx = candidates.set_index("id")
selected_idx = selected_final.set_index("id")

candidate_source_fields = [
    "case",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "final_text_hash"
]

for column in candidate_source_fields:
    assert column in candidate_idx.columns
    assert column in selected_idx.columns

    a = normalized_text(
        candidate_idx[column]
    ).sort_index()

    b = normalized_text(
        selected_idx[column]
    ).sort_index()

    assert a.equals(b), (
        f"Candidate source mismatch: {column}"
    )

assert selected_final["final_text_hash"].is_unique, (
    "Duplicate final text found in model candidates."
)

assert selected_final[
    "duplicate_group_id"
].is_unique, (
    "Multiple model candidates from the same group."
)

assert set(
    selected_final["duplicate_group_id"]
) == set(
    final["duplicate_group_id"]
), (
    "Some text groups lack a model representative."
)

print("CANDIDATE SOURCE MATCH: PASS")
print("ONE REPRESENTATIVE PER GROUP: PASS")
print("MODEL TEXT UNIQUENESS: PASS")

# ------------------------------------------------------------
# L. MODELING HANDOFF DATASET
# ------------------------------------------------------------

# Candidate text and ordering come from the approved
# cleaning output, while labels come from final adjudication.

handoff = candidates.copy()

# Avoid stale annotation fields or merge suffixes.
handoff = handoff.drop(
    columns=[
        column
        for column in ANNOTATION_COLUMNS
        if column in handoff.columns
    ],
    errors="ignore"
)

missing_label_cols = [
    column
    for column in ANNOTATION_COLUMNS
    if column not in final.columns
]

assert not missing_label_cols, (
    "Final labeled dataset missing columns: "
    + str(missing_label_cols)
)

handoff = handoff.merge(
    final[["id"] + ANNOTATION_COLUMNS],
    on="id",
    how="left",
    validate="one_to_one",
    sort=False
)

assert len(handoff) == 348
assert handoff["id"].is_unique

assert handoff["sentiment_label"].isin(
    VALID_LABELS
).all()

assert handoff["final_text_hash"].is_unique

assert normalized_text(
    handoff["label_notes"]
).ne("").all()

assert handoff["annotation_status"].eq(
    "LABELED"
).all()

assert handoff["id"].tolist() == (
    candidates["id"].tolist()
), "Candidate ordering changed unexpectedly."

# Original 381 data and final 348 candidate labels
# remain linked by ID.
handoff_lookup = handoff.set_index("id")

assert normalized_label(
    handoff_lookup["sentiment_label"]
).sort_index().equals(
    normalized_label(
        selected_idx["sentiment_label"]
    ).sort_index()
)

print("348 MODELING CANDIDATES: PASS")

# ------------------------------------------------------------
# M. DISTRIBUTIONS AND IMBALANCE CHECKS
# ------------------------------------------------------------

distribution_381 = count_distribution(final)
distribution_348 = count_distribution(handoff)

case_distribution_381 = {
    str(key): int(value)
    for key, value in final[
        "case"
    ].value_counts().items()
}

case_distribution_348 = {
    str(key): int(value)
    for key, value in handoff[
        "case"
    ].value_counts().items()
}

negative_count = int(
    handoff["sentiment_label"]
    .eq("NEGATIF")
    .sum()
)

negative_share = (
    negative_count / len(handoff)
) * 100

print("\nFINAL LABEL DISTRIBUTION — 381")
display(
    final["sentiment_label"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

print("\nMODEL LABEL DISTRIBUTION — 348")
display(
    handoff["sentiment_label"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

print("\nMODEL DISTRIBUTION BY DISASTER CASE")
display(
    pd.crosstab(
        handoff["case"],
        handoff["sentiment_label"],
        margins=True
    )
)

print(
    f"\nNEGATIF proportion in modeling data: "
    f"{negative_share:.2f}%"
)

if negative_share < 10:
    print(
        "CLASS IMBALANCE WARNING: "
        "NEGATIF is underrepresented."
    )

# ------------------------------------------------------------
# N. PREPARE AUDIT REPORT
# ------------------------------------------------------------

audit = {
    "stage": "03_Labeling",
    "part": "Final Labeling Audit and Modeling Handoff",
    "generated_at_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "master_rows": int(len(master)),
    "final_labeled_rows": int(len(final)),
    "model_candidates": int(len(handoff)),
    "nonselected_rows": int(
        len(final) - len(handoff)
    ),
    "unique_text_groups": int(len(group_sizes)),
    "duplicate_groups": int(
        len(actual_duplicate_ids)
    ),
    "duplicate_rows": int(len(duplicates)),
    "duplicate_conflicts_remaining": 0,
    "reviewed_adjudication_rows": int(
        len(changelog)
    ),
    "changed_labels": int(
        len(actual_changed_ids)
    ),
    "missing_final_labels": 0,
    "missing_annotation_notes": 0,
    "original_master_modified": False,
    "final_distribution_381": distribution_381,
    "model_distribution_348": distribution_348,
    "case_distribution_381": case_distribution_381,
    "case_distribution_348": case_distribution_348,
    "negative_class_share_pct": round(
        negative_share, 4
    ),
    "modeling_split_created": False,
    "modeling_training_started": False,
    "status": "READY_FOR_MODELING_PREPARATION"
}

# ------------------------------------------------------------
# O. FINAL INPUT INTEGRITY CHECK
# ------------------------------------------------------------

input_hashes_after = {
    str(path): sha256_file(path)
    for path in required_inputs
}

assert (
    input_hashes_before == input_hashes_after
), "An input file changed during audit."

audit["input_sha256"] = input_hashes_after

print("\nINPUT HASH INTEGRITY: PASS")
print("MASTER WORKBOOK: UNCHANGED")

# ------------------------------------------------------------
# P. SAVE OUTPUTS
# ------------------------------------------------------------

# Temporary files avoid presenting a partial final file
# as a completed handoff.
TEMP_HANDOFF = FINAL_DIR / ".modeling_labeled_348.tmp.csv"
TEMP_AUDIT = FINAL_DIR / ".final_labeling_audit.tmp.json"

assert not TEMP_HANDOFF.exists()
assert not TEMP_AUDIT.exists()

try:
    handoff.to_csv(
        TEMP_HANDOFF,
        index=False,
        encoding="utf-8-sig"
    )

    # Verify persisted handoff before final publication.
    persisted = read_csv_with_ids(TEMP_HANDOFF)

    assert len(persisted) == EXPECTED_CANDIDATES
    assert persisted["id"].is_unique
    assert persisted["id"].tolist() == (
        candidates["id"].tolist()
    )

    assert persisted[
        "sentiment_label"
    ].isin(VALID_LABELS).all()

    assert persisted[
        "final_text_hash"
    ].is_unique

    audit["modeling_file_sha256"] = (
        sha256_file(TEMP_HANDOFF)
    )

    TEMP_AUDIT.write_text(
        json.dumps(
            audit,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    # Publish only after validations succeed.
    assert not HANDOFF_PATH.exists()
    assert not AUDIT_PATH.exists()

    TEMP_HANDOFF.rename(HANDOFF_PATH)
    TEMP_AUDIT.rename(AUDIT_PATH)

finally:
    # Clean unfinished temporary files only.
    for temp_path in [TEMP_HANDOFF, TEMP_AUDIT]:
        if temp_path.exists():
            temp_path.unlink()

assert sha256_file(HANDOFF_PATH) == (
    audit["modeling_file_sha256"]
)

assert input_hashes_before == {
    str(path): sha256_file(path)
    for path in required_inputs
}

# ------------------------------------------------------------
# Q. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LABELING 13 — FINAL AUDIT SUMMARY")
print("=" * 70)

print("Total labeled tweets     :", len(final))
print("Modeling candidates      :", len(handoff))
print("Nonselected records      :", len(final) - len(handoff))
print("Unique text groups       :", len(group_sizes))
print("Duplicate groups         :", len(actual_duplicate_ids))
print("Duplicate rows           :", len(duplicates))
print("Duplicate label conflicts:", 0)
print("Reviewed adjudications   :", len(changelog))
print("Changed labels           :", len(actual_changed_ids))
print("Missing final labels     :", 0)
print("Missing annotation notes :", 0)

print("\nOutput files:")
print("Model dataset :", HANDOFF_PATH)
print("Audit report  :", AUDIT_PATH)

print("\n" + "=" * 70)
print("LABELING 13 FINAL AUDIT: PASS")
print("348 MODELING CANDIDATES: READY")
print("ORIGINAL MASTER: UNCHANGED")
print("=" * 70)


LABELING 13 — FINAL AUDIT & MODELING HANDOFF
REQUIRED INPUTS: PASS
NO OUTPUT OVERWRITE: PASS
MASTER ROWS: 381
FINAL ROWS : 381
CANDIDATES : 348
CHANGELOG  : 4
ID INTEGRITY: PASS
ALL 381 LABELS: VALID
ANNOTATION NOTES: COMPLETE
ANNOTATOR METADATA: COMPLETE
LABELING STATUS: PASS
MASTER SOURCE INTEGRITY: PASS
ORIGINAL ANNOTATIONS PRESERVED: PASS
ADJUDICATION CHANGELOG: PASS
ACTUAL LABEL CHANGES: 2
UNIQUE TEXT GROUPS: 348
ACTUAL DUPLICATE GROUPS: 21
DUPLICATE ROWS: 54
CONFLICTING LABEL GROUPS: 0
DUPLICATE AUDIT: PASS
CANDIDATE SOURCE MATCH: PASS
ONE REPRESENTATIVE PER GROUP: PASS
MODEL TEXT UNIQUENESS: PASS
348 MODELING CANDIDATES: PASS

FINAL LABEL DISTRIBUTION — 381


,sentiment,count
0,NETRAL,253
1,POSITIF,113
2,NEGATIF,15



MODEL LABEL DISTRIBUTION — 348


,sentiment,count
0,NETRAL,223
1,POSITIF,110
2,NEGATIF,15



MODEL DISTRIBUTION BY DISASTER CASE


sentiment_label,NEGATIF,NETRAL,POSITIF,All
case,,,,
lombok_2018,3,64,34,101
palu_2018,5,70,30,105
sumatra_2025,7,89,46,142
All,15,223,110,348



NEGATIF proportion in modeling data: 4.31%
CLASS IMBALANCE WARNING: NEGATIF is underrepresented.

INPUT HASH INTEGRITY: PASS
MASTER WORKBOOK: UNCHANGED

LABELING 13 — FINAL AUDIT SUMMARY
Total labeled tweets     : 381
Modeling candidates      : 348
Nonselected records      : 33
Unique text groups       : 348
Duplicate groups         : 21
Duplicate rows           : 54
Duplicate label conflicts: 0
Reviewed adjudications   : 4
Changed labels           : 2
Missing final labels     : 0
Missing annotation notes : 0

Output files:
Model dataset : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\final_labeling\modeling_labeled_348.csv
Audit report  : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\final_labeling\final_labeling_audit.json

LABELING 13 FINAL AUDIT: PASS
348 MODELING CANDIDATES: READY
ORIGINAL MASTER: UNCHANGED
